# Лабораторная работа №11: Alignment VLM по предпочтениям

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Собрать preference pairs без length/style shortcut.
- Провести LoRA preference optimization на 2B fallback.
- Сравнить base/SFT/aligned по win rate и hallucinations.


## Теоретическая часть

Preference optimization использует пары chosen/rejected. DPO увеличивает относительную вероятность chosen относительно reference policy без отдельной reward model. Для VLM важно, чтобы изображения и prompts совпадали в обеих ветвях.


### Математическая постановка

$$\mathcal L_{DPO}=-\log\sigma\left(\beta[\log\tfrac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)}-\log\tfrac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}]\right).$$


### Материалы

- [DPO paper](https://arxiv.org/abs/2305.18290)
- [TRL цикл оптимизации DPO](https://huggingface.co/docs/trl/dpo_trainer)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Preference dataset является экспертным входом; пример с несуществующим cats.jpg не используется. Нужен адаптер lab 9 и GPU. При смене версии TRL сверьте мультимодальный цикл оптимизации DPO и зафиксируйте версии. `FAST_DEV_RUN=True` выполняет реальные два шага DPO, но не является evidence качества. Без заполненного слепого экспертного аудита итоговая метрика не создаётся.

In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path(os.getenv('CV_ARTIFACTS', 'artifacts/lab11')); ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path(os.getenv('CV_DATA', 'data/lab9')); DATA_DIR.mkdir(parents=True, exist_ok=True)  # те же изображения, что в работе 09
LAB9_ARTIFACTS = Path('artifacts/lab9')
LAB9_ADAPTER = LAB9_ARTIFACTS/'lab9_adapter'  # SFT-адаптер из работы 09; если его нет — fallback (см. задание о политиках)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
from pathlib import Path
import json, hashlib
from datasets import load_dataset

SCIENCEQA_ID = 'derek-thomas/ScienceQA'
SCIENCEQA_REVISION = 'f18b0a70359ebfb41f658fd564208d0355b013f4'

def prepare_scienceqa(directory=None, smoke=False):
    """Fixed official splits, image-level deduplication, explicitly defined shifts."""
    directory = DATA_DIR if directory is None else directory
    directory = Path(directory).resolve(); directory.mkdir(parents=True, exist_ok=True)
    manifest = directory / ('manifest-smoke.jsonl' if smoke else 'manifest.jsonl')
    limits = {'train': 8 if smoke else 512, 'validation': 4 if smoke else 64,
              'id_test': 4 if smoke else 64, 'ood_test': 4 if smoke else 64,
              'general_test': 4 if smoke else 64}
    if manifest.exists():
        records = [json.loads(s) for s in manifest.read_text(encoding='utf-8').splitlines()]
        if all(Path(r['image']).is_file() for r in records) and all(sum(r['split']==s for r in records)==n for s,n in limits.items()):
            return manifest
    records, seen, counts = [], set(), dict.fromkeys(limits, 0)
    for official in ('train', 'validation', 'test'):
        data = load_dataset(SCIENCEQA_ID, revision=SCIENCEQA_REVISION, split=official, streaming=True)
        for index, row in enumerate(data):
            image = row['image']
            if image is None: continue
            if official != 'test':
                if row['subject'] != 'natural science' or int(row['grade'].removeprefix('grade')) > 6: continue
                split = official
            elif row['subject'] != 'natural science': split = 'general_test'
            elif int(row['grade'].removeprefix('grade')) > 6: split = 'ood_test'
            else: split = 'id_test'
            if counts[split] >= limits[split]: continue
            image = image.convert('RGB')
            digest = hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
            if digest in seen: continue
            seen.add(digest)
            path = directory / f'{digest}.png'; image.save(path)
            answer = int(row['answer']); choices = row['choices']
            question = row['question']+'\n'+'\n'.join(f'{chr(65+i)}. {v}' for i,v in enumerate(choices))+'\nReturn only the letter of the correct option.'
            records.append({'id': f'scienceqa-{official}-{index}', 'split': split,
                            'image': str(path), 'question': question, 'answer': chr(65+answer),
                            'choices': choices, 'answer_index': answer, 'label_type': 'multiple_choice',
                            'license': 'MIT (code license, lupantech/ScienceQA)',
                            'source_url': f'https://huggingface.co/datasets/{SCIENCEQA_ID}/tree/{SCIENCEQA_REVISION}',
                            'subject':row['subject'], 'grade':row['grade'], 'image_sha256':digest})
            counts[split] += 1
            required = [official] if official != 'test' else ['id_test','ood_test','general_test']
            if all(counts[s] == limits[s] for s in required): break
    if counts != limits: raise RuntimeError(f'Insufficient licensed image examples: {counts}; expected {limits}')
    manifest.write_text('\n'.join(json.dumps(r,ensure_ascii=False) for r in records)+'\n', encoding='utf-8')
    (directory/'ATTRIBUTION.md').write_text('ScienceQA — Lu et al., NeurIPS 2022. Dataset distribution: '+SCIENCEQA_ID+'\nRevision: '+SCIENCEQA_REVISION+'\nDataset-card license: CC-BY-SA-4.0, https://creativecommons.org/licenses/by-sa/4.0/\nOriginal project: https://scienceqa.github.io/\nImage subsets and manifests retain attribution; downloaded images are not committed to this course.\nID: natural science grades 1–6; OOD: natural science grades 7–12; general: other subjects. This is an operational shift, not a claim of universal model ability.\n', encoding='utf-8')
    return manifest


## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Задание 1. Предпочтения из визуальных ответов

*Вес: 1 балл.*

**Постановка.** Подготовьте ScienceQA через открытый загрузчик. Для каждого изображения chosen — правильная буква, rejected — следующий неверный вариант. Явно пометьте: это предпочтения на основе эталонной разметки, а не экспертные оценки полезности.

**Результат.** Список пар с provenance.

**Критерий принятия.** Изображения существуют, ответы различаются и соответствуют вариантам; лицензия и источник сохранены.


Создаем таблицу учебных примеров. Каждая строка выглядит так:
```text
картинка:  фото с рыбой и выдрой
вопрос:    Which organism...? A. black rockfish  B. sea otter
chosen:    A    ← так отвечать хорошо
rejected:  B    ← так отвечать плохо
provenance: откуда взят пример: датасет, версия, лицензия, способ получения пары
```
Пара — это два ответа на один вопрос: хороший и плохой.<br>
Provenance — «паспорт» примера. Он показывает, откуда взяты данные, и что пары получены из готовых правильных ответов, а не от экспертов.

Пары нужны для обучения методом DPO: модели показывают хороший и плохой ответ и учат предпочитать хороший.
- SFT (лаба 9) учит: «на этот вопрос правильный ответ — A».
- DPO (лаба 11) учит: «A лучше, чем B».

**Задание 1. Предпочтения из визуальных ответов**

**Что делали**: из манифеста ScienceQA (те же изображения, что в работе 09) собрали пары для каждого вопроса: chosen — правильная буква, rejected — следующий вариант по кругу (после последнего снова A). К каждой паре приложили provenance: датасет, ревизию, лицензию и пометку `reference_label_derived`.<br>
**Зачем**: получить обучающие пары «хороший / плохой ответ» для DPO. Пометка показывает, что предпочтения выведены из правильных ответов датасета, а не из оценок экспертов.<br>
**Результат**: 768 пар (train 512, validation 64, id_test 64, ood_test 64, general_test 64). Все изображения на месте, sha256 совпадает, chosen ≠ rejected, обе буквы есть среди вариантов, лицензия и источник сохранены. Файл `preference_pairs_check.json`.

In [2]:
# Предпочтения из визуальных ответов.
# ВАЖНО: это предпочтения на основе эталонной разметки ScienceQA (correct vs. следующий неверный вариант),
# а НЕ экспертные оценки полезности ответа.
from PIL import Image

PREFERENCE_SOURCE = 'reference_label_derived'  # не human/expert preference
manifest_path = prepare_scienceqa(smoke=FAST_DEV_RUN)
records = [json.loads(s) for s in manifest_path.read_text(encoding='utf-8').splitlines() if s.strip()]

def letter(i):
    return chr(65 + i)

def make_pair(r):
    n, ai = len(r['choices']), r['answer_index']
    ri = (ai + 1) % n  # следующий по кругу вариант; при n >= 2 он всегда неверный
    return {
        'id': r['id'], 'split': r['split'],
        'image': r['image'], 'image_sha256': r['image_sha256'],
        'prompt': r['question'],
        'chosen': letter(ai), 'rejected': letter(ri),
        'chosen_text': r['choices'][ai], 'rejected_text': r['choices'][ri],
        'chosen_index': ai, 'rejected_index': ri,
        'choices': r['choices'],
        'subject': r['subject'], 'grade': r['grade'],
        'provenance': {
            'preference_type': PREFERENCE_SOURCE,
            'note': 'chosen = эталонная буква ScienceQA, rejected = следующий неверный вариант; '
                    'это не экспертные оценки полезности',
            'dataset': SCIENCEQA_ID, 'revision': SCIENCEQA_REVISION,
            'source_url': r['source_url'],
            'license': r['license'],
            'dataset_license': 'CC-BY-SA-4.0 (dataset card), https://creativecommons.org/licenses/by-sa/4.0/',
            'manifest': str(manifest_path),
        },
    }

preference_pairs = [make_pair(r) for r in records if len(r['choices']) >= 2]

# --- Критерии принятия ---
problems = []
for p in preference_pairs:
    path = Path(p['image'])
    if not path.is_file():
        problems.append((p['id'], 'image missing')); continue
    img = Image.open(path).convert('RGB')
    if hashlib.sha256(str(img.size).encode() + img.tobytes()).hexdigest() != p['image_sha256']:
        problems.append((p['id'], 'image hash mismatch'))
    letters = [letter(i) for i in range(len(p['choices']))]
    if not p['chosen'] or not p['rejected'] or p['chosen'] == p['rejected']:
        problems.append((p['id'], 'empty or identical answers'))
    if p['chosen'] not in letters or p['rejected'] not in letters:
        problems.append((p['id'], 'answer not among options'))
    if f"{p['chosen']}. {p['chosen_text']}" not in p['prompt'] or f"{p['rejected']}. {p['rejected_text']}" not in p['prompt']:
        problems.append((p['id'], 'letter/text mismatch with prompt'))
    if p['chosen_index'] != next(r['answer_index'] for r in records if r['id'] == p['id']):
        problems.append((p['id'], 'chosen is not the reference answer'))
    if not p['provenance']['license'] or not p['provenance']['source_url']:
        problems.append((p['id'], 'license/source missing'))
assert (manifest_path.parent / 'ATTRIBUTION.md').is_file(), 'ATTRIBUTION.md не найден'
assert len(preference_pairs) == len(records), 'есть записи с < 2 вариантами'
assert not problems, problems[:10]

from collections import Counter
pairs_check = {
    'preference_type': PREFERENCE_SOURCE,
    'n_pairs': len(preference_pairs),
    'by_split': dict(Counter(p['split'] for p in preference_pairs)),
    'n_choices': dict(Counter(len(p['choices']) for p in preference_pairs)),
    'all_images_exist_and_hash_ok': True,
    'all_answers_distinct_and_valid': True,
    'license_and_source_saved': True,
    'attribution_file': str(manifest_path.parent / 'ATTRIBUTION.md'),
}
save_json('preference_pairs_check.json', pairs_check)
print(json.dumps(pairs_check, ensure_ascii=False, indent=2))
print('Пример пары:', json.dumps({k: preference_pairs[0][k] for k in ('id', 'chosen', 'rejected', 'chosen_text', 'rejected_text')}, ensure_ascii=False))

{
  "preference_type": "reference_label_derived",
  "n_pairs": 768,
  "by_split": {
    "train": 512,
    "validation": 64,
    "ood_test": 64,
    "general_test": 64,
    "id_test": 64
  },
  "n_choices": {
    "2": 392,
    "3": 291,
    "4": 78,
    "5": 7
  },
  "all_images_exist_and_hash_ok": true,
  "all_answers_distinct_and_valid": true,
  "license_and_source_saved": true,
  "attribution_file": "C:\\Users\\bogom\\OneDrive\\Документы\\PSY\\6 курс\\МО\\data\\lab9\\ATTRIBUTION.md"
}
Пример пары: {"id": "scienceqa-train-9", "chosen": "A", "rejected": "B", "chosen_text": "black rockfish", "rejected_text": "sea otter"}


### Задание 2. Отложенный тест и смещения пар

*Вес: 1 балл.*

**Постановка.** Используйте official train для DPO и ID test для итоговой оценки. Сверьте ID и хеши изображений. Посчитайте распределение типов ошибок и долю случаев, где chosen длиннее rejected.

**Результат.** preferences.jsonl и pair_audit.json.

**Критерий принятия.** Нет пересечений; длина ответа не используется как скрытый эталон правильности.


**Задание 2. Отложенный тест и смещения пар**

**Что делали**: разделили пары: official train (512) — для DPO, id_test (64) — для итоговой оценки. Сверили ID и хеши изображений, посчитали распределение типов ошибок и долю пар, где текст правильного варианта длиннее неверного.<br>
**Зачем**: убедиться, что тест не пересекается с обучением и что длина ответа не подсказывает правильный вариант.<br>
**Результат**: пересечений train/id_test 0 (по ID, хешу, пути и паре «картинка + вопрос»). Типы ошибок в train: второй вариант из двух 43,4 %, другой правдоподобный 27,3 %, противоположный вывод 19,1 %, «одинаково» 9,6 %, неверное число 0,6 %. chosen и rejected — одна буква (длина совпадает в 100 % пар). Текст правильного варианта длиннее в 37,3 % / 34,4 % пар (train / id_test). Правило «выбирай самый длинный» даёт 38,7 % / 35,2 %, хуже случайного выбора (42,7 % / 42,6 %). Файлы `preferences.jsonl`, `pair_audit.json`.

In [3]:
# Отложенный тест и смещения пар.
# official train -> DPO, ID test -> итоговая оценка. Остальные сплиты в эту работу не входят.
import re
from collections import Counter

train_pairs = [p for p in preference_pairs if p['split'] == 'train']
test_pairs = [p for p in preference_pairs if p['split'] == 'id_test']
assert train_pairs and test_pairs, 'пустой train или id_test'

# --- 1. Сверка ID и хешей изображений ---
by_id = {r['id']: r for r in records}
for p in train_pairs + test_pairs:
    r = by_id[p['id']]
    assert p['image_sha256'] == r['image_sha256'] == Path(p['image']).stem, f"хеш не совпадает: {p['id']}"
    assert p['id'].startswith('scienceqa-train-' if p['split'] == 'train' else 'scienceqa-test-'), \
        f"ID не из официального сплита: {p['id']}"
assert len({p['id'] for p in train_pairs}) == len(train_pairs), 'дубли ID в train'
assert len({p['id'] for p in test_pairs}) == len(test_pairs), 'дубли ID в id_test'

def norm_q(p):
    return re.sub(r'\s+', ' ', p['prompt'].lower()).strip()

overlap = {
    'id': len({p['id'] for p in train_pairs} & {p['id'] for p in test_pairs}),
    'image_sha256': len({p['image_sha256'] for p in train_pairs} & {p['image_sha256'] for p in test_pairs}),
    'image_path': len({p['image'] for p in train_pairs} & {p['image'] for p in test_pairs}),
    'image+question': len({(p['image_sha256'], norm_q(p)) for p in train_pairs}
                          & {(p['image_sha256'], norm_q(p)) for p in test_pairs}),
}
# Справочно: совпадение текста вопроса без изображения допустимо (шаблонные вопросы ScienceQA)
question_text_overlap = len({norm_q(p) for p in train_pairs} & {norm_q(p) for p in test_pairs})
assert not any(overlap.values()), f'пересечение train/id_test: {overlap}'

# --- 2. Типы ошибок (какой неверный вариант стал rejected) ---
FAMILIES = [
    ('magnet_force', r'magnetic force'),
    ('magnet_attract_repel', r'attract or repel'),
    ('object_property', r'which property'),
    ('weather_vs_climate', r'weather or the climate'),
    ('kinetic_energy_temperature', r'kinetic energ|temperature'),
    ('solution_concentration', r'concentration'),
    ('organism_classification', r'^select the .* below|scientific name|same genus'),
    ('animal_adaptation', r'adapted'),
    ('experiment_variable', r'variable in this experiment'),
    ('molecule_chemistry', r'molecule|chemical formula|elementary substance|compound'),
    ('force_direction', r'direction of this (pull|push)'),
    ('diagram_text_completion', r'^complete the'),
]
OPPOSITES = [('attract', 'repel'), ('weather', 'climate'), ('true', 'false'), ('higher', 'lower'),
             ('stronger', 'weaker'), ('pair 1', 'pair 2'), ('solution a', 'solution b'),
             ('sample a', 'sample b'), ('toward', 'away'), ('increase', 'decrease')]

def task_family(p):
    q = p['prompt'].split('\n')[0].lower()
    return next((name for name, pat in FAMILIES if re.search(pat, q)), 'other')

def distractor_kind(p):
    c, r = p['chosen_text'].lower(), p['rejected_text'].lower()
    if any((a in c and b in r) or (b in c and a in r) for a, b in OPPOSITES):
        return 'opposite_of_correct'          # противоположный вывод (attract/repel, Pair 1/Pair 2, ...)
    if 'same' in r or 'neither' in r or 'same' in c or 'neither' in c:
        return 'no_difference_vs_difference'  # «одинаково» против конкретного ответа
    if re.search(r'\d', c) and re.search(r'\d', r):
        return 'wrong_numeric_value'
    if len(p['choices']) == 2:
        return 'binary_other_option'
    return 'plausible_distractor'             # другой правдоподобный вариант из списка

for p in train_pairs + test_pairs:
    p['task_family'] = task_family(p)
    p['error_type'] = distractor_kind(p)

def dist(pairs, key):
    return dict(Counter(p[key] for p in pairs).most_common())

def dist_share(pairs, key):
    return {k: round(v / len(pairs), 4) for k, v in dist(pairs, key).items()}

ERROR_TYPE_RU = {
    'binary_other_option':         'второй вариант из двух',
    'plausible_distractor':        'другой правдоподобный вариант',
    'opposite_of_correct':         'противоположный вывод',
    'no_difference_vs_difference': '«одинаково» vs конкретный ответ',
    'wrong_numeric_value':         'неверное числовое значение',
}

LABEL_W, COL_W = 48, 14

def table_row(label, train_cell, test_cell):
    print(f"  {label:<{LABEL_W}}{train_cell:>{COL_W}} │{test_cell:>{COL_W}}")

def table_header(title):
    print(f"\n{title}")
    table_row('', 'train', 'id_test')
    print('  ' + '─' * LABEL_W + '─' * COL_W + '─┼' + '─' * COL_W)

def count_cell(k, n):
    return f'{k} ({k / n:6.1%})'

def print_error_types():
    tr, te = Counter(p['error_type'] for p in train_pairs), Counter(p['error_type'] for p in test_pairs)
    table_header('Распределение типов ошибок (rejected):')
    for k, _ in (tr + te).most_common():
        table_row(ERROR_TYPE_RU.get(k, k), count_cell(tr[k], len(train_pairs)), count_cell(te[k], len(test_pairs)))
    table_row('всего', count_cell(len(train_pairs), len(train_pairs)), count_cell(len(test_pairs), len(test_pairs)))

# --- 3. Смещение по длине ---
# DPO-цели — одиночные буквы, поэтому длина chosen и rejected совпадает по построению.
assert all(len(p['chosen']) == len(p['rejected']) == 1 for p in train_pairs + test_pairs), \
    'chosen/rejected должны быть одной буквой'

def length_stats(pairs):
    longer = sum(len(p['chosen_text']) > len(p['rejected_text']) for p in pairs)
    shorter = sum(len(p['chosen_text']) < len(p['rejected_text']) for p in pairs)
    # Бейзлайн «выбрать самый длинный вариант» среди всех вариантов (ничья делится поровну)
    longest_hits = 0.0
    for p in pairs:
        lens = [len(t) for t in p['choices']]
        winners = [i for i, l in enumerate(lens) if l == max(lens)]
        longest_hits += (p['chosen_index'] in winners) / len(winners)
    chance = sum(1 / len(p['choices']) for p in pairs) / len(pairs)
    return {
        'n': len(pairs),
        'answer_token_len_equal_share': 1.0,                 # буква vs буква
        'chosen_text_longer_share': round(longer / len(pairs), 4),
        'chosen_text_shorter_share': round(shorter / len(pairs), 4),
        'equal_text_length_share': round((len(pairs) - longer - shorter) / len(pairs), 4),
        'longest_option_baseline_acc': round(longest_hits / len(pairs), 4),
        'random_baseline_acc': round(chance, 4),
    }

length_bias = {'train': length_stats(train_pairs), 'id_test': length_stats(test_pairs)}

# --- 4. Артефакты ---
def write_jsonl(path, rows):
    path.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')

for p in train_pairs: p['use'] = 'dpo_train'
for p in test_pairs:  p['use'] = 'final_eval_id_test'
write_jsonl(ARTIFACTS / 'preferences.jsonl', train_pairs + test_pairs)

pair_audit = {
    'preference_type': PREFERENCE_SOURCE,
    'note': 'Предпочтения выведены из эталонной разметки ScienceQA, это не экспертные оценки полезности.',
    'dataset': SCIENCEQA_ID, 'revision': SCIENCEQA_REVISION,
    'split_usage': {'train': 'DPO (official train)', 'id_test': 'итоговая оценка (official test, ID)'},
    'counts': {'train': len(train_pairs), 'id_test': len(test_pairs)},
    'id_and_hash_check': 'ok: ID из официальных сплитов, sha256 совпадает с манифестом и именем файла',
    'overlap_train_vs_id_test': overlap,
    'question_text_only_overlap_info': question_text_overlap,
    'task_family_distribution': {'train': dist(train_pairs, 'task_family'), 'id_test': dist(test_pairs, 'task_family')},
    'error_type_distribution': {'train': dist(train_pairs, 'error_type'), 'id_test': dist(test_pairs, 'error_type')},
    'error_type_share': {'train': dist_share(train_pairs, 'error_type'), 'id_test': dist_share(test_pairs, 'error_type')},
    'task_family_share': {'train': dist_share(train_pairs, 'task_family'), 'id_test': dist_share(test_pairs, 'task_family')},
    'n_choices_distribution': {
        'train': dict(Counter(len(p['choices']) for p in train_pairs)),
        'id_test': dict(Counter(len(p['choices']) for p in test_pairs)),
    },
    'length_bias': length_bias,
    'length_not_hidden_label': (
        'chosen и rejected — одиночные буквы равной длины, поэтому длина ответа не несёт сигнала '
        'о правильности. Доля пар, где текст правильного варианта длиннее, указана справочно; '
        'бейзлайн «самый длинный вариант» сравнивается со случайным.'
    ),
}
save_json('pair_audit.json', pair_audit)

print('preferences.jsonl:', len(train_pairs), 'train +', len(test_pairs), 'id_test')
print('Пересечения train/id_test:', overlap, '| совпадений только текста вопроса:', question_text_overlap)
print_error_types()
table_header('Смещение по длине:')
rows = [
    ('chosen и rejected одной длины (буква vs буква)', 'answer_token_len_equal_share'),
    ('текст chosen ДЛИННЕЕ текста rejected',          'chosen_text_longer_share'),
    ('текст chosen КОРОЧЕ текста rejected',           'chosen_text_shorter_share'),
    ('тексты одинаковой длины',                       'equal_text_length_share'),
    ('точность «выбрать самый длинный вариант»',      'longest_option_baseline_acc'),
    ('точность случайного выбора',                    'random_baseline_acc'),
]
for label, key in rows:
    table_row(label, f"{length_bias['train'][key]:.1%}", f"{length_bias['id_test'][key]:.1%}")
print('  Вывод: «самый длинный вариант» не лучше случайного -> длина не является скрытым эталоном правильности.'
      if all(length_bias[s]['longest_option_baseline_acc'] <= length_bias[s]['random_baseline_acc'] + 0.05 for s in length_bias)
      else '  ВНИМАНИЕ: длинный вариант заметно чаще правильный -> возможен length shortcut в текстах вариантов.')

preferences.jsonl: 512 train + 64 id_test
Пересечения train/id_test: {'id': 0, 'image_sha256': 0, 'image_path': 0, 'image+question': 0} | совпадений только текста вопроса: 12

Распределение типов ошибок (rejected):
                                                           train │       id_test
  ───────────────────────────────────────────────────────────────┼──────────────
  второй вариант из двух                            222 ( 43.4%) │   25 ( 39.1%)
  другой правдоподобный вариант                     140 ( 27.3%) │   18 ( 28.1%)
  противоположный вывод                              98 ( 19.1%) │   14 ( 21.9%)
  «одинаково» vs конкретный ответ                    49 (  9.6%) │    7 ( 10.9%)
  неверное числовое значение                          3 (  0.6%) │    0 (  0.0%)
  всего                                             512 (100.0%) │   64 (100.0%)

Смещение по длине:
                                                           train │       id_test
  ──────────────────────────────────

**Распределение типов ошибок.** Таблица показывает, какими бывают неправильные ответы (rejected) в парах. Модель учится отличать правильный ответ именно от таких ошибок.

| Тип ошибки | Условный пример (chosen → rejected) | train | id_test |
|---|---|---:|---:|
| Второй вариант из двух | всего два варианта, rejected — другой | 222 (43,4 %) | 25 (39,1 %) |
| Другой правдоподобный вариант | «гладкий» → «твёрдый» | 140 (27,3 %) | 18 (28,1 %) |
| Противоположный вывод | «оттолкнутся» → «притянутся» | 98 (19,1 %) | 14 (21,9 %) |
| «Одинаково» против конкретного ответа | «в паре 1 сильнее» → «равны» | 49 (9,6 %) | 7 (10,9 %) |
| Неверное число | 25 °C → 8 °C | 3 (0,6 %) | 0 (0,0 %) |
| Всего | | 512 (100 %) | 64 (100 %) |

**Смещение по длине**
Это проверка, не подсказывает ли длина ответа, какой из них правильный.<br>
Модель ищет самый простой способ отличить chosen от rejected. Допустим, правильный ответ почти всегда длиннее. Тогда модель выучит правило «выбирай, что длиннее» и не будет смотреть на картинку.

### Задание 3. Начальная и фиксированная политики

*Вес: 1 балл.*

**Постановка.** Создайте обучаемый адаптер и отдельный фиксированный reference. Используйте адаптер работы 09, если он доступен; иначе обе политики начинаются с опубликованной instruction-tuned модели. Запишите выбранный источник.

**Результат.** reference_policy.json.

**Критерий принятия.** Начальные политики совпадают; параметры reference не обновляются; запуск без адаптера 09 не называется продолжением собственного SFT.


Берем адаптер из 9 лабораторной. Её будем сравнивать с обученой на DPO

**Задание 3. Начальная и фиксированная политики**

**Что делали**: загрузили Qwen2.5-VL-3B-Instruct в 4-bit NF4 и подключили к ней два адаптера из работы 09: `default` (обучаемая политика) и `ref` (замороженный reference). Разрешили обучение только LoRA-весам `default`.<br>
**Зачем**: DPO сравнивает обучаемую модель с её неизменной исходной копией, поэтому на старте нужны две одинаковые политики, одна из которых никогда не меняется.<br>
**Результат**: источник — `lab09_sft_adapter`, это продолжение собственного SFT. Обучается 14 966 784 параметра. Веса `default` и `ref` совпадают между собой и с файлом работы 09 (max |Δ| = 0). На проверочной паре log p у обеих одинаковый: chosen −0,0070 (≈ 99,3 %), rejected −5,2570 (≈ 0,5 %). Файл `reference_policy.json`.

In [4]:
# Начальная и фиксированная политики.
# Одна 4-bit base-модель и два LoRA-адаптера поверх неё (соглашение TRL 1.x о PEFT + DPO):
#   'default' — обучаемая политика π_θ, 'ref' — замороженная reference-политика π_ref.
# Если адаптер работы 09 доступен, ОБА адаптера загружаются из него (продолжение собственного SFT);
# иначе обе политики начинаются с опубликованной Qwen2.5-VL-3B-Instruct (новый LoRA с нулевой B-матрицей).
import math, importlib.metadata as md
import torch
from PIL import Image
from safetensors.torch import load_file
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from transformers.image_utils import SizeDict
from peft import PeftModel, LoraConfig, get_peft_model

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU не найден: DPO для VLM на CPU не выполняется')

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
POLICY, REF = 'default', 'ref'
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# --- 1. Выбор источника начальной политики ---
lab9_cfg_path = LAB9_ARTIFACTS / 'lab9_adapter.run_config.json'
lab9_files_ok = all((LAB9_ADAPTER / f).is_file() for f in ('adapter_config.json', 'adapter_model.safetensors'))
lab9_base = json.loads((LAB9_ADAPTER / 'adapter_config.json').read_text(encoding='utf-8'))['base_model_name_or_path'] \
    if lab9_files_ok else None
USE_LAB9 = lab9_files_ok and lab9_base == MODEL_ID
SOURCE = 'lab09_sft_adapter' if USE_LAB9 else 'published_instruct_model'
LAB9_RUN = json.loads(lab9_cfg_path.read_text(encoding='utf-8'))['config'] if USE_LAB9 and lab9_cfg_path.is_file() else {}
# Те же условия, что при SFT в работе 09: 4-bit nf4 + double quant, лимит пикселей изображения
MAX_PIXELS = LAB9_RUN.get('image_max_pixels', 512 * 512)

def load_image(path):
    """RGB, площадь не больше MAX_PIXELS с сохранением пропорций (как image_max_pixels в работе 09)."""
    img = Image.open(path).convert('RGB')
    w, h = img.size
    if w * h > MAX_PIXELS:
        s = math.sqrt(MAX_PIXELS / (w * h))
        img = img.resize((max(28, int(w * s)), max(28, int(h * s))), Image.BICUBIC)
    return img

# --- 2. Base-модель и processor ---
torch.manual_seed(SEED)
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
                                bnb_4bit_compute_dtype=DTYPE)
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, dtype=DTYPE, device_map={'': 0})
processor = AutoProcessor.from_pretrained(MODEL_ID)
ip = processor.image_processor
ip.size = SizeDict(shortest_edge=ip.size['shortest_edge'], longest_edge=MAX_PIXELS)
for p in base_model.parameters():          # base (включая vision tower) не обучается
    p.requires_grad_(False)

# --- 3. Обучаемая политика и отдельный фиксированный reference ---
if USE_LAB9:
    policy_model = PeftModel.from_pretrained(base_model, str(LAB9_ADAPTER), adapter_name=POLICY, is_trainable=True)
    policy_model.load_adapter(str(LAB9_ADAPTER), adapter_name=REF, is_trainable=False)  # независимая загрузка из файла
else:
    lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                          target_modules=r'.*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)')
    policy_model = get_peft_model(base_model, lora_cfg, adapter_name=POLICY)    # B = 0 -> политика == base
    policy_model.add_adapter(REF, lora_cfg)
    for n, p in policy_model.named_parameters():
        if f'.{POLICY}.' in n:
            policy_model.get_parameter(n.replace(f'.{POLICY}.', f'.{REF}.')).data.copy_(p.data)
policy_model.set_adapter(POLICY)

def freeze_reference(model):
    """Обучаются только LoRA-веса политики; reference и base заморожены.
    PEFT.set_adapter() включает requires_grad у активного адаптера, поэтому заморозку применяем явно."""
    for n, p in model.named_parameters():
        p.requires_grad_(f'.{POLICY}.' in n and 'lora_' in n)

freeze_reference(policy_model)

def adapter_params(model, name):
    return {n.replace(f'.{name}.', '.'): p for n, p in model.named_parameters() if f'.{name}.' in n}

def params_sha256(params):
    h = hashlib.sha256()
    for n in sorted(params):
        h.update(n.encode()); h.update(params[n].detach().float().cpu().numpy().tobytes())
    return h.hexdigest()

pol, ref = adapter_params(policy_model, POLICY), adapter_params(policy_model, REF)

# --- 4. Проверки критериев принятия ---
# 4a. Начальные политики совпадают по весам
assert pol.keys() == ref.keys() and len(pol) > 0, 'разный набор LoRA-тензоров у policy и reference'
max_weight_diff = max((pol[k].detach().float() - ref[k].detach().float()).abs().max().item() for k in pol)
assert max_weight_diff == 0.0, f'начальные веса policy и reference различаются: {max_weight_diff}'

# 4b. Оба адаптера совпадают с файлом адаптера работы 09 (все тензоры загружены, ничего не пропущено)
if USE_LAB9:
    file_sd = load_file(str(LAB9_ADAPTER / 'adapter_model.safetensors'))
    file_sd = {k.removeprefix('base_model.model.'): v for k, v in file_sd.items()}
    loaded = {k.removeprefix('base_model.model.'): v for k, v in pol.items()}
    assert set(file_sd) == set(loaded), \
        f'ключи адаптера 09 не совпали: нет {len(set(file_sd) - set(loaded))}, лишних {len(set(loaded) - set(file_sd))}'
    lab9_file_diff = max((file_sd[k].float() - loaded[k].detach().float().cpu()).abs().max().item() for k in file_sd)
    assert lab9_file_diff < 1e-6, f'загруженный адаптер отличается от файла работы 09: {lab9_file_diff}'
else:
    lab9_file_diff = None
    assert all(v.detach().abs().max().item() == 0 for k, v in pol.items() if 'lora_B' in k), 'новый LoRA должен иметь B = 0'

# 4c. Reference не обновляется: ни один его параметр не требует градиента
trainable = {n: p for n, p in policy_model.named_parameters() if p.requires_grad}
assert trainable and all(f'.{POLICY}.' in n for n in trainable), 'обучаются параметры вне адаптера политики'
assert not any(p.requires_grad for p in ref.values()), 'reference требует градиент'
REF_SHA256 = params_sha256(ref)    # контрольная сумма: после DPO reference должен остаться таким же

# 4d. Функциональная проверка: одинаковый log p(chosen|x) и log p(rejected|x) у политики и reference
def build_example(pair, answer):
    """Изображение + вопрос (как в работе 09), затем ответ-буква; маска только на токены ответа."""
    messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': pair['prompt']}]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    img = load_image(pair['image'])
    full = processor(text=[prompt + answer], images=[img], return_tensors='pt')
    n_prompt = processor(text=[prompt], images=[img], return_tensors='pt')['input_ids'].shape[1]
    return full, n_prompt

@torch.inference_mode()
def answer_logp(model, pair, answer, adapter):
    """Сумма log-вероятностей токенов ответа при активном адаптере adapter."""
    model.set_adapter(adapter)
    try:
        enc, n_prompt = build_example(pair, answer)
        enc = enc.to(model.device)
        logits = model(**enc).logits[0, :-1].float()
        targets = enc['input_ids'][0, 1:]
        logp = torch.log_softmax(logits, -1).gather(-1, targets[:, None])[:, 0]
        return logp[n_prompt - 1:].sum().item()
    finally:
        model.set_adapter(POLICY)
        freeze_reference(model)

policy_model.eval()
check_pair = train_pairs[0]
logp_check = {}
for adapter in (POLICY, REF):
    logp_check[adapter] = {'chosen': answer_logp(policy_model, check_pair, check_pair['chosen'], adapter),
                           'rejected': answer_logp(policy_model, check_pair, check_pair['rejected'], adapter)}
max_logp_diff = max(abs(logp_check[POLICY][k] - logp_check[REF][k]) for k in ('chosen', 'rejected'))
assert max_logp_diff < 1e-4, f'начальные политики дают разные log p: {max_logp_diff}'
assert params_sha256(adapter_params(policy_model, REF)) == REF_SHA256   # проверка не изменила reference

# --- 5. Артефакт ---
n_trainable = sum(p.numel() for p in trainable.values())
n_ref = sum(p.numel() for p in ref.values())
reference_policy = {
    'source': SOURCE,
    'continuation_of_own_sft': USE_LAB9,
    'source_note': ('Обе политики загружены из SFT-адаптера работы 09 (LLaMA-Factory, LoRA r=8, alpha=16): '
                    'DPO продолжает собственный SFT.') if USE_LAB9 else
                   ('Адаптер работы 09 недоступен: обе политики начинаются с опубликованной instruction-tuned модели '
                    f'{MODEL_ID} (новый LoRA, B = 0). Это НЕ продолжение собственного SFT.'),
    'base_model': MODEL_ID,
    'quantization': {'load_in_4bit': True, 'quant_type': 'nf4', 'double_quant': True, 'compute_dtype': str(DTYPE)},
    'image_max_pixels': MAX_PIXELS,
    'lab09_adapter': {
        'path': str(LAB9_ADAPTER), 'available': lab9_files_ok, 'base_model': lab9_base,
        'adapter_file_sha256': hashlib.sha256((LAB9_ADAPTER / 'adapter_model.safetensors').read_bytes()).hexdigest()
                               if lab9_files_ok else None,
        'run_config_fingerprint': json.loads(lab9_cfg_path.read_text(encoding='utf-8')).get('fingerprint')
                                  if lab9_cfg_path.is_file() else None,
    },
    'adapters': {
        'policy': {'name': POLICY, 'trainable': True, 'n_params': n_trainable,
                   'lora_r': policy_model.peft_config[POLICY].r, 'lora_alpha': policy_model.peft_config[POLICY].lora_alpha},
        'reference': {'name': REF, 'trainable': False, 'n_params': n_ref, 'params_sha256': REF_SHA256},
    },
    'checks': {
        'initial_policies_equal_weights': True, 'max_abs_weight_diff': max_weight_diff,
        'loaded_equals_lab09_file_max_abs_diff': lab9_file_diff,
        'reference_requires_grad_any': False,
        'trainable_only_policy_lora': True,
        'logp_check': {'pair_id': check_pair['id'], **logp_check, 'max_abs_diff': max_logp_diff},
    },
    'trl_note': 'TRL 1.x: обучаемый адаптер — "default", reference — "ref"; DPOTrainer считает π_ref с активным "ref". '
                'После обучения REF_SHA256 сверяется повторно.',
    'versions': {p: md.version(p) for p in ('torch', 'transformers', 'peft', 'trl', 'bitsandbytes')},
}
save_json('reference_policy.json', reference_policy)

print(f'Источник начальной политики: {SOURCE}  (продолжение собственного SFT: {USE_LAB9})')
print(f'Обучаемых параметров (policy LoRA): {n_trainable:,} | параметров reference (заморожены): {n_ref:,}')
print(f'Веса policy vs reference: max |Δ| = {max_weight_diff}' +
      (f' | vs файл адаптера 09: max |Δ| = {lab9_file_diff:.1e}' if USE_LAB9 else ''))
print(f"log p на {check_pair['id']}: policy chosen={logp_check[POLICY]['chosen']:.4f}, rejected={logp_check[POLICY]['rejected']:.4f} | "
      f"ref chosen={logp_check[REF]['chosen']:.4f}, rejected={logp_check[REF]['rejected']:.4f}")
print('reference sha256:', REF_SHA256[:16], '…  -> artifacts/lab11/reference_policy.json')

W1101 16:05:57.409000 29436 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

c:\Users\bogom\OneDrive\Документы\PSY\6 курс\МО\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Источник начальной политики: lab09_sft_adapter  (продолжение собственного SFT: True)
Обучаемых параметров (policy LoRA): 14,966,784 | параметров reference (заморожены): 14,966,784
Веса policy vs reference: max |Δ| = 0.0 | vs файл адаптера 09: max |Δ| = 0.0e+00
log p на scienceqa-train-9: policy chosen=-0.0070, rejected=-5.2570 | ref chosen=-0.0070, rejected=-5.2570
reference sha256: bfc294e34c6c062d …  -> artifacts/lab11/reference_policy.json


### Задание 4. Log-prob мультимодального ответа

*Вес: 1 балл.*

**Постановка.** Реализуйте prepare_answer и completion_logprob. Сохраните все image tensors процессора, исключите prompt из суммы логарифмов вероятностей и учитывайте только токены ответа и окончания.

**Результат.** Функции подготовки и scoring.

**Критерий принятия.** Префикс токенов full совпадает с отдельно токенизированным prompt; выбран правильный сдвиг next-token labels; score дифференцируем по policy.


Подгатавливаем инструменты измерения.

`prepare_answer` готовит входные данные. Она собирает картинку, вопрос и ответ в понятный модели вид и размечает, где заканчивается вопрос и начинается ответ:
```text
[картинка + вопрос: 389 токенов]  [A] [<|im_end|>] [перевод строки]
 маска: 0 (не считаем)             1       1             1
```
Токены <|im_end|> и перевод строки — это окончание реплики модели. Задание требует учитывать и их.

`completion_logprob` считает вероятность. Она прогоняет всё это через модель и складывает логарифмы вероятностей только токенов ответа. Вопрос в сумму не входит: нас интересует вероятность ответа, а не вопроса.

**Задание 4. Log-prob мультимодального ответа**

**Что делали**: написали `prepare_answer` (картинка + вопрос + ответ, маска токенов ответа, все image-тензоры процессора) и `completion_logprob` (сумма log p только по токенам ответа и окончания со сдвигом next-token).<br>
**Зачем**: формула DPO построена на log-вероятностях ответов, поэтому нужен точный и дифференцируемый способ их считать.<br>
**Результат**: prompt — 389 токенов, ответ + окончание — `A`, `<|im_end|>`, перевод строки. Префикс совпадает с отдельно токенизированным prompt, сохранены `pixel_values`, `image_grid_thw`, `mm_token_type_ids`. Score −0,0070 совпал со встроенным loss модели (без сдвига было бы −79,88). Градиент есть только у LoRA `default` (504 тензора, норма 35,0). Наборы данных: train 512, eval 64. Файл `logprob_check.json`.

In [5]:
# Log-prob мультимодального ответа.
# prepare_answer: prompt (изображение + вопрос) и full = prompt + ответ + токены окончания реплики,
#                 все тензоры процессора (pixel_values, image_grid_thw, ...) сохраняются.
# completion_logprob: Σ log p(токен | префикс) только по токенам ответа и окончания, prompt исключён.
from datasets import Dataset
from trl.trainer.dpo_trainer import DataCollatorForVisionPreference

TEXT_KEYS = ('input_ids', 'attention_mask')

def prompt_messages(pair):
    """Тот же формат, что в работе 09 и в reference-проверке: сначала изображение, затем вопрос."""
    return [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': pair['prompt']}]}]

def answer_message(answer):
    return [{'role': 'assistant', 'content': [{'type': 'text', 'text': answer}]}]

def prepare_answer(pair, answer, image=None):
    """Мультимодальные входы для scoring ответа `answer` на пару `pair`.

    Возвращает dict: input_ids/attention_mask для full, ВСЕ image-тензоры процессора,
    completion_mask (1 — токены ответа и окончания реплики), prompt_len и completion_text.
    """
    image = load_image(pair['image']) if image is None else image
    prompt_text = processor.apply_chat_template(prompt_messages(pair), tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(prompt_messages(pair) + answer_message(answer), tokenize=False)
    assert full_text.startswith(prompt_text), 'chat template отрисовал prompt по-разному'
    completion_text = full_text[len(prompt_text):]          # 'A<|im_end|>\n' — ответ и окончание

    # add_special_tokens=False: шаблон уже содержит спецтокены (как в TRL)
    prompt_enc = processor(text=[prompt_text], images=[image], return_tensors='pt', add_special_tokens=False)
    full_enc = processor(text=[full_text], images=[image], return_tensors='pt', add_special_tokens=False)

    prompt_len = prompt_enc['input_ids'].shape[1]
    # Критерий: префикс токенов full совпадает с отдельно токенизированным prompt
    assert torch.equal(full_enc['input_ids'][0, :prompt_len], prompt_enc['input_ids'][0]), 'prompt не является префиксом full'
    assert full_enc['input_ids'].shape[1] > prompt_len, 'пустой ответ'
    # Все image-тензоры одинаковы в prompt и full (одно изображение, одна обработка)
    image_keys = [k for k in full_enc if k not in TEXT_KEYS]
    for k in image_keys:
        if k == 'mm_token_type_ids':
            assert torch.equal(full_enc[k][0, :prompt_len], prompt_enc[k][0]), k
        else:
            assert torch.equal(full_enc[k], prompt_enc[k]), f'image-тензор {k} различается'

    completion_mask = torch.zeros_like(full_enc['input_ids'])
    completion_mask[:, prompt_len:] = 1
    return {**dict(full_enc), 'completion_mask': completion_mask, 'prompt_len': prompt_len,
            'completion_text': completion_text, 'image_keys': image_keys}

def model_inputs(inputs, device):
    return {k: v.to(device) for k, v in inputs.items() if torch.is_tensor(v) and k != 'completion_mask'}

def completion_logprob(model, inputs, return_tokens=False):
    """log π(ответ | изображение, вопрос) = Σ_{t ∈ ответ} log p(x_t | x_<t).

    Сдвиг next-token: логиты позиции t-1 предсказывают токен t, поэтому logits[:, :-1] ↔ input_ids[:, 1:]
    и completion_mask[:, 1:]. Без torch.no_grad — результат дифференцируем по обучаемой политике.
    """
    batch = model_inputs(inputs, model.device)
    logits = model(**batch).logits                                  # [B, T, V]
    targets = batch['input_ids'][:, 1:]                             # токен t
    mask = inputs['completion_mask'][:, 1:].to(model.device).bool() # t — токен ответа/окончания
    pred_logits = logits[:, :-1][mask].float()                      # логиты позиции t-1 (только нужные)
    token_logp = torch.log_softmax(pred_logits, dim=-1).gather(-1, targets[mask][:, None])[:, 0]
    seq_ids = mask.nonzero()[:, 0]
    score = torch.zeros(mask.shape[0], device=model.device, dtype=token_logp.dtype).index_add(0, seq_ids, token_logp)
    return (score, token_logp) if return_tokens else score

# --- Проверки критериев принятия ---
policy_model.eval()
policy_model.set_adapter(POLICY); freeze_reference(policy_model)
pair0 = train_pairs[0]
img0 = load_image(pair0['image'])
inp_c = prepare_answer(pair0, pair0['chosen'], img0)
inp_r = prepare_answer(pair0, pair0['rejected'], img0)
tok = processor.tokenizer
completion_tokens = tok.convert_ids_to_tokens(inp_c['input_ids'][0, inp_c['prompt_len']:])
assert inp_c['prompt_len'] == inp_r['prompt_len']
assert tok.eos_token in inp_c['completion_text'] or '<|im_end|>' in inp_c['completion_text'], 'нет токена окончания'

# 1) Правильный сдвиг: наш score совпадает с встроенным loss модели (он сдвигает labels сам)
with torch.no_grad():
    score_c, tok_logp_c = completion_logprob(policy_model, inp_c, return_tokens=True)
    labels = inp_c['input_ids'].clone(); labels[inp_c['completion_mask'] == 0] = -100
    hf_loss = policy_model(**model_inputs(inp_c, policy_model.device), labels=labels.to(policy_model.device)).loss
    n_completion = int(inp_c['completion_mask'].sum())
    hf_score = -hf_loss.float().item() * n_completion
    # Для сравнения — неправильный вариант без сдвига (логиты позиции t для токена t)
    logits = policy_model(**model_inputs(inp_c, policy_model.device)).logits[0].float()
    m = inp_c['completion_mask'][0].bool().to(logits.device)
    unshifted = torch.log_softmax(logits[m], -1).gather(-1, inp_c['input_ids'][0].to(logits.device)[m][:, None]).sum().item()
shift_diff = abs(score_c.item() - hf_score)
assert shift_diff < 1e-2, f'сдвиг next-token неверен: {score_c.item()} vs {hf_score}'
assert abs(unshifted - score_c.item()) > 1.0, 'проверка не различает сдвинутый и несдвинутый вариант'

# 2) Score дифференцируем по policy: градиент есть у LoRA 'default', нет у 'ref' и base
policy_model.zero_grad(set_to_none=True)
score_r = completion_logprob(policy_model, inp_r)
assert score_r.requires_grad, 'score не связан с графом вычислений'
score_r.sum().backward()
grads = {n: p.grad for n, p in policy_model.named_parameters() if p.grad is not None}
policy_grad_norm = torch.sqrt(sum((g.float() ** 2).sum() for g in grads.values())).item()
assert grads and all(f'.{POLICY}.' in n for n in grads), 'градиент попал вне адаптера политики'
assert policy_grad_norm > 0, 'нулевой градиент по политике'
n_grad_tensors = len(grads)
policy_model.zero_grad(set_to_none=True)
assert params_sha256(adapter_params(policy_model, REF)) == REF_SHA256   # reference не тронут

# 3) Согласованность с коллатором TRL (то, что увидит DPOTrainer)
def to_trl_example(pair, image):
    return {'id': pair['id'], 'images': [image],
            'prompt': [{'role': 'user', 'content': pair['prompt']}],
            'chosen': [{'role': 'assistant', 'content': pair['chosen']}],
            'rejected': [{'role': 'assistant', 'content': pair['rejected']}]}

trl_batch = DataCollatorForVisionPreference(processor)([to_trl_example(pair0, img0)])
for row, ours in ((0, inp_c), (1, inp_r)):
    keep = trl_batch['attention_mask'][row].bool()
    assert torch.equal(trl_batch['input_ids'][row][keep], ours['input_ids'][0]), 'токены TRL ≠ prepare_answer'
    assert torch.equal(trl_batch['completion_mask'][row][keep], ours['completion_mask'][0]), 'маска TRL ≠ наша'
assert torch.equal(trl_batch['pixel_values'][: len(inp_c['pixel_values'])], inp_c['pixel_values'])

# --- Артефакт: мультимодальный Dataset с изображениями и парными ответами ---
def build_dataset(pairs):
    return Dataset.from_list([to_trl_example(p, load_image(p['image'])) for p in pairs])

dpo_train_dataset = build_dataset(train_pairs[:8] if FAST_DEV_RUN else train_pairs)
dpo_eval_dataset = build_dataset(test_pairs[:4] if FAST_DEV_RUN else test_pairs)
assert dpo_train_dataset[0]['chosen'][0]['content'] != dpo_train_dataset[0]['rejected'][0]['content']

logprob_check = {
    'pair_id': pair0['id'],
    'prompt_len_tokens': inp_c['prompt_len'],
    'completion_text_chosen': inp_c['completion_text'],
    'completion_tokens_chosen': completion_tokens,
    'image_tensors_kept': inp_c['image_keys'],
    'prefix_matches_prompt': True,
    'score_chosen': round(score_c.item(), 6), 'score_rejected': round(score_r.item(), 6),
    'token_logp_chosen': [round(x, 6) for x in tok_logp_c.tolist()],
    'shift_check': {'ours': round(score_c.item(), 6), 'hf_loss_based': round(hf_score, 6),
                    'abs_diff': shift_diff, 'unshifted_wrong': round(unshifted, 6)},
    'grad_check': {'policy_grad_norm': policy_grad_norm, 'n_grad_tensors': n_grad_tensors,
                   'only_policy_lora_has_grad': True, 'reference_unchanged': True},
    'trl_collator_consistent': True,
    'dataset': {'train': len(dpo_train_dataset), 'eval_id_test': len(dpo_eval_dataset),
                'columns': dpo_train_dataset.column_names},
}
save_json('logprob_check.json', logprob_check)

print(f"Пара {pair0['id']}: prompt = {inp_c['prompt_len']} токенов, ответ+окончание = {completion_tokens}")
print(f"Image-тензоры процессора сохранены: {inp_c['image_keys']}")
print(f"Префикс full == prompt: да | токены и маска совпадают с коллатором TRL: да")
print(f"Score chosen = {score_c.item():.4f}, rejected = {score_r.item():.4f}  (log p по токенам ответа: "
      f"{', '.join(f'{x:.4f}' for x in tok_logp_c.tolist())})")
print(f"Сдвиг next-token: наш = {score_c.item():.4f}, через loss модели = {hf_score:.4f} (|Δ| = {shift_diff:.1e}); "
      f"без сдвига было бы {unshifted:.2f}")
print(f"Градиент: норма по LoRA политики = {policy_grad_norm:.3e} ({n_grad_tensors} тензоров), у reference и base — нет")
print(f"Dataset: train = {len(dpo_train_dataset)}, eval (id_test) = {len(dpo_eval_dataset)}, колонки {dpo_train_dataset.column_names}")

c:\Users\bogom\OneDrive\Документы\PSY\6 курс\МО\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Пара scienceqa-train-9: prompt = 389 токенов, ответ+окончание = ['A', '<|im_end|>', 'Ċ']
Image-тензоры процессора сохранены: ['mm_token_type_ids', 'pixel_values', 'image_grid_thw']
Префикс full == prompt: да | токены и маска совпадают с коллатором TRL: да
Score chosen = -0.0070, rejected = -5.2570  (log p по токенам ответа: -0.0070, -0.0000, 0.0000)
Сдвиг next-token: наш = -0.0070, через loss модели = -0.0070 (|Δ| = 0.0e+00); без сдвига было бы -79.88
Градиент: норма по LoRA политики = 3.501e+01 (504 тензоров), у reference и base — нет
Dataset: train = 512, eval (id_test) = 64, колонки ['id', 'images', 'prompt', 'chosen', 'rejected']


**Проверка критериев**

| Критерий | Что это значит | Результат |
|---|---|---|
| Префикс совпадает | начало полной последовательности — это ровно отдельно обработанный вопрос; иначе граница ответа определится неверно | совпали все 389 токенов |
| Правильный сдвиг | модель на каждой позиции предсказывает **следующий** токен, поэтому предсказания и токены сопоставляются со сдвигом на один | совпало со встроенным расчётом модели (−0,0070); без сдвига вышло бы −79,88 |
| Score дифференцируем | из результата можно посчитать градиент, а значит, обучать модель | градиент есть только у `default`, у `ref` и Qwen его нет |

### Задание 5. Целевая функция DPO

*Вес: 1 балл.*

**Постановка.** Настройте beta=0.1, AdamW lr=5e-6, accumulation=8 и 40 optimizer steps. Реализуйте отрицательный log-sigmoid разности policy/ref margins; в optimizer включите только trainable adapter.

**Результат.** dpo_protocol.json и optimizer.

**Критерий принятия.** Рост policy-margin при фиксированном reference уменьшает loss; вычисление reference идёт без градиентов.


Настраивается оптимизатор и функция потерь

**Задание 5. Целевая функция DPO**

**Что делали**: реализовали loss DPO `−log σ(β · (policy_margin(обучаемая модель. Вероятность ответа chosen минус вероятность rejected) − ref_margin(тоже самое у фиксированной модели)))`, настроили β = 0,1, AdamW lr = 5·10⁻⁶, accumulation = 8, 40 optimizer steps и включили в optimizer только обучаемый адаптер.<br>
**Зачем**: задать правило обучения и проверить его до запуска: рост margin политики должен уменьшать loss, а reference должен считаться без градиентов.<br>
**Результат**: 40 шагов × 8 пар = 320 пар (0,625 эпохи). В optimizer 504 тензора, 14 966 784 параметра — только `default`. При фиксированном ref_margin = 2,5 loss убывает: policy_margin −10 → 1,50; 2,5 → 0,69; 10 → 0,39. На реальной паре loss = 0,6931 = log 2 (политика = reference). sha256 reference не изменился. Файл `dpo_protocol.json`.

In [6]:
# Целевая функция DPO.
# L = −log σ(β · [(log π_θ(y_w|x) − log π_ref(y_w|x)) − (log π_θ(y_l|x) − log π_ref(y_l|x))])
#   = −log σ(β · (policy_margin − ref_margin)),  margin = log p(chosen) − log p(rejected)
# Свой компактный DPO-тренер вместо trl.DPOTrainer: TRL 1.x для 4-bit модели переводит обучаемый LoRA в bf16,
# а адаптер 09 хранится в fp32 — начальная политика перестала бы точно совпадать с reference.
# Формула loss та же, что у TRL (loss_type='sigmoid'): -F.logsigmoid(beta * delta_score).
import torch.nn.functional as F

DPO = dict(beta=0.1, learning_rate=5e-6, gradient_accumulation_steps=8, max_steps=2 if FAST_DEV_RUN else 40,
           micro_batch_pairs=1, optimizer='AdamW', adam_betas=(0.9, 0.999), adam_eps=1e-8, weight_decay=0.0,
           lr_schedule='constant', max_grad_norm=1.0, disable_dropout=True, gradient_checkpointing=True)
DPO['effective_batch_pairs'] = DPO['micro_batch_pairs'] * DPO['gradient_accumulation_steps']
DPO['pairs_seen'] = DPO['max_steps'] * DPO['effective_batch_pairs']

def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=DPO['beta']):
    """Отрицательный log-sigmoid разности policy/ref margins (по последовательностям)."""
    policy_margin = policy_chosen - policy_rejected
    ref_margin = ref_chosen - ref_rejected
    logits = beta * (policy_margin - ref_margin)
    loss = -F.logsigmoid(logits)
    # неявные награды DPO: β·log(π_θ/π_ref) для chosen и rejected
    rewards = {'chosen': beta * (policy_chosen - ref_chosen).detach(),
               'rejected': beta * (policy_rejected - ref_rejected).detach()}
    return loss, policy_margin, ref_margin, rewards


class AdapterDPOTrainer:
    """DPO на одной 4-bit модели с двумя LoRA: обучаемый POLICY и замороженный REF."""

    def __init__(self, model, policy_adapter, ref_adapter, cfg):
        self.model, self.policy, self.ref, self.cfg = model, policy_adapter, ref_adapter, cfg
        self.model.set_adapter(self.policy); freeze_reference(self.model)
        if cfg['gradient_checkpointing']:
            self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
            self.model.enable_input_require_grads()
        # Режим train обязателен: в transformers gradient checkpointing работает только при model.training=True
        # (в eval он молча выключается и активации всех слоёв держатся в памяти)
        self.model.train()
        if cfg['disable_dropout']:   # dropout выключаем обнулением p (как disable_dropout_in_model в TRL)
            for module in self.model.modules():
                if isinstance(module, torch.nn.Dropout):
                    module.p = 0.0
        # В optimizer — только обучаемый адаптер
        self.trainable = [p for n, p in model.named_parameters() if p.requires_grad]
        self.optimizer = torch.optim.AdamW(self.trainable, lr=cfg['learning_rate'], betas=cfg['adam_betas'],
                                           eps=cfg['adam_eps'], weight_decay=cfg['weight_decay'])

    @torch.no_grad()
    def reference_logps(self, inp_c, inp_r):
        """log π_ref без градиентов: активен замороженный адаптер REF."""
        self.model.set_adapter(self.ref)
        try:
            return completion_logprob(self.model, inp_c), completion_logprob(self.model, inp_r)
        finally:
            self.model.set_adapter(self.policy); freeze_reference(self.model)

    def pair_loss(self, pair):
        image = load_image(pair['image'])
        inp_c, inp_r = prepare_answer(pair, pair['chosen'], image), prepare_answer(pair, pair['rejected'], image)
        ref_c, ref_r = self.reference_logps(inp_c, inp_r)
        assert not ref_c.requires_grad and not ref_r.requires_grad, 'reference вычислен с градиентом'
        self.model.set_adapter(self.policy)
        pol_c, pol_r = completion_logprob(self.model, inp_c), completion_logprob(self.model, inp_r)
        loss, pm, rm, rewards = dpo_loss(pol_c, pol_r, ref_c, ref_r, self.cfg['beta'])
        return loss.mean(), {'policy_chosen': pol_c.item(), 'policy_rejected': pol_r.item(),
                             'ref_chosen': ref_c.item(), 'ref_rejected': ref_r.item(),
                             'policy_margin': pm.item(), 'ref_margin': rm.item(),
                             'reward_chosen': rewards['chosen'].item(), 'reward_rejected': rewards['rejected'].item()}

    def optimizer_step(self, pairs):
        """Один optimizer step = gradient_accumulation_steps пар; loss делится на число микробатчей."""
        assert len(pairs) == self.cfg['gradient_accumulation_steps']
        self.optimizer.zero_grad(set_to_none=True)
        losses, stats = [], []
        for pair in pairs:
            loss, st = self.pair_loss(pair)
            (loss / len(pairs)).backward()
            losses.append(loss.item()); stats.append(st)
        grad_norm = torch.nn.utils.clip_grad_norm_(self.trainable, self.cfg['max_grad_norm']).item()
        self.optimizer.step()
        self.optimizer.zero_grad(set_to_none=True)
        return {'loss': float(np.mean(losses)), 'grad_norm': grad_norm, 'pairs': [p['id'] for p in pairs],
                'policy_margin': float(np.mean([s['policy_margin'] for s in stats])),
                'ref_margin': float(np.mean([s['ref_margin'] for s in stats])),
                'reward_chosen': float(np.mean([s['reward_chosen'] for s in stats])),
                'reward_rejected': float(np.mean([s['reward_rejected'] for s in stats])),
                'reward_accuracy': float(np.mean([s['reward_chosen'] > s['reward_rejected'] for s in stats]))}


dpo_trainer = AdapterDPOTrainer(policy_model, POLICY, REF, DPO)
optimizer = dpo_trainer.optimizer

# Фиксированный порядок пар для всех optimizer steps (перестановки train по эпохам с SEED)
order_rng = random.Random(SEED)
DATA_ORDER = []
while len(DATA_ORDER) < DPO['pairs_seen']:
    epoch = list(range(len(train_pairs))); order_rng.shuffle(epoch); DATA_ORDER += epoch
DATA_ORDER = DATA_ORDER[:DPO['pairs_seen']]

# --- Проверки критериев принятия ---
# 1) В optimizer только обучаемый адаптер
opt_params = {id(p) for g in optimizer.param_groups for p in g['params']}
policy_lora = {id(p) for n, p in policy_model.named_parameters() if f'.{POLICY}.' in n and 'lora_' in n}
ref_ids = {id(p) for p in adapter_params(policy_model, REF).values()}
assert opt_params == policy_lora, 'optimizer содержит не только LoRA политики'
assert not opt_params & ref_ids, 'reference попал в optimizer'
n_opt_params = sum(p.numel() for g in optimizer.param_groups for p in g['params'])

# 2) Рост policy-margin при фиксированном reference уменьшает loss (аналитически и через autograd)
ref_c0, ref_r0 = torch.tensor(-0.5), torch.tensor(-3.0)                 # фиксированный reference
margins = torch.linspace(-20, 20, 81)
grid_loss = torch.stack([dpo_loss(torch.tensor(0.) + m, torch.tensor(0.), ref_c0, ref_r0)[0] for m in margins])
assert torch.all(grid_loss[1:] < grid_loss[:-1]), 'loss не убывает с ростом policy-margin'
m = torch.tensor(1.0, requires_grad=True)
dpo_loss(m, torch.tensor(0.), ref_c0, ref_r0)[0].backward()
assert m.grad.item() < 0, 'dL/d(policy_margin) должен быть < 0'
# при policy == reference loss = log 2
assert abs(dpo_loss(ref_c0, ref_r0, ref_c0, ref_r0)[0].item() - math.log(2)) < 1e-6

# 3) Реальная пара: reference без градиентов, градиент только у политики; шаг optimizer НЕ делается
assert policy_model.training and all(m.p == 0.0 for m in policy_model.modules() if isinstance(m, torch.nn.Dropout))
optimizer.zero_grad(set_to_none=True)
loss0, st0 = dpo_trainer.pair_loss(train_pairs[DATA_ORDER[0]])
assert loss0.requires_grad
loss0.backward()
grads = {n for n, p in policy_model.named_parameters() if p.grad is not None}
assert grads and all(f'.{POLICY}.' in n for n in grads), 'градиент вне адаптера политики'
assert abs(loss0.item() - math.log(2)) < 1e-3, f'на старте policy == reference, loss должен быть log 2: {loss0.item()}'
optimizer.zero_grad(set_to_none=True)
assert params_sha256(adapter_params(policy_model, REF)) == REF_SHA256, 'reference изменился'

# --- Артефакт ---
dpo_protocol = {
    'fast_dev_run': FAST_DEV_RUN,
    'loss': '-log_sigmoid(beta * ((logp_pol(y_w) - logp_pol(y_l)) - (logp_ref(y_w) - logp_ref(y_l))))',
    'logp': 'Σ log p по токенам ответа и окончания (completion_logprob), prompt исключён',
    'config': {k: (list(v) if isinstance(v, tuple) else v) for k, v in DPO.items()},
    'policy_adapter': POLICY, 'reference_adapter': REF,
    'reference_source': SOURCE, 'continuation_of_own_sft': USE_LAB9, 'reference_sha256': REF_SHA256,
    'optimizer': {'type': type(optimizer).__name__, 'n_tensors': len(opt_params), 'n_params': n_opt_params,
                  'contains_only_policy_lora': True, **{k: (list(v) if isinstance(v, tuple) else v)
                  for k, v in optimizer.defaults.items() if k in ('lr', 'betas', 'eps', 'weight_decay')}},
    'data': {'train_pairs': len(train_pairs), 'pairs_seen': DPO['pairs_seen'],
             'epochs': round(DPO['pairs_seen'] / len(train_pairs), 3),
             'order_sha256': hashlib.sha256(json.dumps([train_pairs[i]['id'] for i in DATA_ORDER]).encode()).hexdigest()},
    'checks': {
        'loss_decreases_with_policy_margin': True,
        'dloss_dmargin_at_1': m.grad.item(),
        'reference_no_grad': True,
        'only_policy_lora_gets_grad': True,
        'initial_loss_real_pair': {'pair_id': train_pairs[DATA_ORDER[0]]['id'], 'loss': loss0.item(),
                                   'expected': math.log(2), **st0},
        'reference_unchanged': True,
    },
    'trainer': 'AdapterDPOTrainer (своя реализация, формула = TRL loss_type="sigmoid")',
}
save_json('dpo_protocol.json', dpo_protocol)

print(f"DPO: beta={DPO['beta']}, AdamW lr={DPO['learning_rate']}, accumulation={DPO['gradient_accumulation_steps']}, "
      f"optimizer steps={DPO['max_steps']} -> {DPO['pairs_seen']} пар ({dpo_protocol['data']['epochs']} эпохи train)")
print(f"В optimizer: {len(opt_params)} тензоров, {n_opt_params:,} параметров — только LoRA '{POLICY}', reference '{REF}' не входит")
print('Loss при росте policy-margin (reference фиксирован):')
for mv in (-10, -5, 0, 2.5, 5, 10):
    lv = dpo_loss(torch.tensor(float(mv)), torch.tensor(0.), ref_c0, ref_r0)[0].item()
    print(f'  policy_margin = {mv:>5}  (ref_margin = {(ref_c0 - ref_r0).item():.1f})  ->  loss = {lv:.4f}')
print(f"Реальная пара {train_pairs[DATA_ORDER[0]]['id']}: loss = {loss0.item():.4f} (log 2 = {math.log(2):.4f}), "
      f"policy_margin = {st0['policy_margin']:.4f}, ref_margin = {st0['ref_margin']:.4f}")
print('Reference: считается под torch.no_grad, градиентов нет, sha256 не изменился -> artifacts/lab11/dpo_protocol.json')

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


DPO: beta=0.1, AdamW lr=5e-06, accumulation=8, optimizer steps=40 -> 320 пар (0.625 эпохи train)
В optimizer: 504 тензоров, 14,966,784 параметров — только LoRA 'default', reference 'ref' не входит
Loss при росте policy-margin (reference фиксирован):
  policy_margin =   -10  (ref_margin = 2.5)  ->  loss = 1.5019
  policy_margin =    -5  (ref_margin = 2.5)  ->  loss = 1.1369
  policy_margin =     0  (ref_margin = 2.5)  ->  loss = 0.8259
  policy_margin =   2.5  (ref_margin = 2.5)  ->  loss = 0.6931
  policy_margin =     5  (ref_margin = 2.5)  ->  loss = 0.5759
  policy_margin =    10  (ref_margin = 2.5)  ->  loss = 0.3869
Реальная пара scienceqa-train-1203: loss = 0.6931 (log 2 = 0.6931), policy_margin = 6.6250, ref_margin = 6.6250
Reference: считается под torch.no_grad, градиентов нет, sha256 не изменился -> artifacts/lab11/dpo_protocol.json


### Задание 6. Обучение предпочтениям

*Вес: 1 балл.*

**Постановка.** Выполните реальное DPO-обучение, сохраняя loss каждого шага. Не маскируйте NaN. Сохраните обучаемый LoRA в отдельный каталог.

**Результат.** aligned_adapter/train и dpo_metrics.json.

**Критерий принятия.** Обучение выполнило заданный бюджет; loss конечен, адаптер сохранён; smoke не считается полным экспериментом.

**Задание 6. Обучение предпочтениям**

**Что делали**: провели реальное DPO-обучение адаптера `default`: 40 шагов по 8 пар, loss каждого шага записан, при NaN обучение остановилось бы с ошибкой.<br>
**Зачем**: получить aligned-политику и проверить, что обучение выполнило бюджет и не сломалось.<br>
**Результат**: выполнено 40 / 40 шагов (320 пар) за 18,4 мин на RTX 3060, пик памяти 3,4 ГБ. Loss 0,693 → 0,608, все значения конечны. Разница policy margin − ref margin выросла с 0 до 1,80, reward accuracy на последнем шаге 1,00. Адаптер сохранён в `aligned_adapter/train/`, reference не изменился. Файлы `dpo_metrics.json`, `dpo_steps_train.jsonl`.

In [7]:
# Обучение предпочтениям: реальное DPO, loss каждого шага, NaN не маскируется.
# Полный режим -> artifacts/lab11/aligned_adapter/train + dpo_metrics.json
# FAST_DEV_RUN -> artifacts/lab11/aligned_adapter/smoke + dpo_metrics_smoke.json (не полный эксперимент)
FORCE_RETRAIN = False   # True — переобучить, даже если завершённый запуск уже сохранён

RUN_KIND = 'smoke' if FAST_DEV_RUN else 'train'
ADAPTER_DIR = ARTIFACTS / 'aligned_adapter' / RUN_KIND
METRICS_PATH = ARTIFACTS / ('dpo_metrics_smoke.json' if FAST_DEV_RUN else 'dpo_metrics.json')
STEP_LOG = ARTIFACTS / f'dpo_steps_{RUN_KIND}.jsonl'          # построчный лог: переживает падение ядра
ACC = DPO['gradient_accumulation_steps']

def policy_state():
    return {n: p for n, p in adapter_params(policy_model, POLICY).items()}

def save_metrics(status, history, extra=None):
    losses = [h['loss'] for h in history]
    metrics = {
        'status': status,
        'run_kind': RUN_KIND,
        'fast_dev_run': FAST_DEV_RUN,
        'is_full_experiment': (not FAST_DEV_RUN) and status == 'completed' and len(history) == 40,
        'note': 'smoke-прогон проверяет механику и НЕ является полным экспериментом' if FAST_DEV_RUN else
                'полный эксперимент: 40 optimizer steps × 8 пар',
        'budget': {'planned_optimizer_steps': DPO['max_steps'], 'done_optimizer_steps': len(history),
                   'gradient_accumulation_steps': ACC, 'pairs_planned': DPO['pairs_seen'],
                   'pairs_done': sum(len(h['pairs']) for h in history)},
        'config': dpo_protocol['config'],
        'reference_source': SOURCE, 'continuation_of_own_sft': USE_LAB9,
        'all_losses_finite': all(math.isfinite(l) for l in losses),
        'loss_first': losses[0] if losses else None, 'loss_last': losses[-1] if losses else None,
        'loss_mean_first5': float(np.mean(losses[:5])) if losses else None,
        'loss_mean_last5': float(np.mean(losses[-5:])) if losses else None,
        'history': history,
        **(extra or {}),
    }
    save_json(METRICS_PATH.name, metrics)
    return metrics

prev = json.loads(METRICS_PATH.read_text(encoding='utf-8')) if METRICS_PATH.is_file() else None
can_reuse = (not FORCE_RETRAIN and prev is not None and prev.get('status') == 'completed'
             and prev['budget']['done_optimizer_steps'] == DPO['max_steps']
             and (ADAPTER_DIR / 'adapter_model.safetensors').is_file())

if globals().get('DPO_TRAINED_IN_SESSION'):
    raise RuntimeError('Политика в этой сессии уже обучена: перезапустите ядро и выполните ячейки сначала, '
                       'иначе DPO продолжится с уже изменённых весов')

if can_reuse:
    # Завершённый запуск уже есть: загружаем сохранённые веса в адаптер политики вместо повторного обучения
    saved = load_file(str(ADAPTER_DIR / 'adapter_model.safetensors'))
    saved = {k.removeprefix('base_model.model.'): v for k, v in saved.items()}
    cur = {k.removeprefix('base_model.model.'): p for k, p in policy_state().items()}
    assert set(saved) == set(cur), 'набор тензоров сохранённого адаптера не совпадает'
    with torch.no_grad():
        for k, p in cur.items():
            p.copy_(saved[k].to(p.device, p.dtype))
    assert params_sha256(policy_state()) == prev['adapter']['policy_params_sha256'], 'загружен не тот адаптер'
    dpo_metrics, history = prev, prev['history']
    print(f'Загружен сохранённый завершённый запуск: {ADAPTER_DIR}  (FORCE_RETRAIN=True — переобучить)')
else:
    STEP_LOG.write_text('', encoding='utf-8')
    history = []
    torch.manual_seed(SEED)
    torch.cuda.reset_peak_memory_stats()
    t_start = time.time()
    status = 'interrupted'
    try:
        for step in range(DPO['max_steps']):
            t0 = time.time()
            pairs = [train_pairs[i] for i in DATA_ORDER[step * ACC:(step + 1) * ACC]]
            optimizer.zero_grad(set_to_none=True)
            losses, stats = [], []
            for pair in pairs:
                loss, st = dpo_trainer.pair_loss(pair)
                if not torch.isfinite(loss):                      # NaN/inf не маскируем — останавливаемся
                    status = 'failed_non_finite_loss'
                    raise FloatingPointError(f'шаг {step + 1}: loss = {loss.item()} на паре {pair["id"]}')
                (loss / ACC).backward()
                losses.append(loss.item()); stats.append(st)
            grad_norm = torch.nn.utils.clip_grad_norm_(dpo_trainer.trainable, DPO['max_grad_norm']).item()
            if not math.isfinite(grad_norm):
                status = 'failed_non_finite_grad'
                raise FloatingPointError(f'шаг {step + 1}: grad_norm = {grad_norm}')
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            rec = {
                'step': step + 1, 'loss': float(np.mean(losses)), 'loss_per_pair': losses,
                'grad_norm_before_clip': grad_norm, 'lr': optimizer.param_groups[0]['lr'],
                'policy_margin': float(np.mean([s['policy_margin'] for s in stats])),
                'ref_margin': float(np.mean([s['ref_margin'] for s in stats])),
                'reward_chosen': float(np.mean([s['reward_chosen'] for s in stats])),
                'reward_rejected': float(np.mean([s['reward_rejected'] for s in stats])),
                'reward_margin': float(np.mean([s['reward_chosen'] - s['reward_rejected'] for s in stats])),
                'reward_accuracy': float(np.mean([s['reward_chosen'] > s['reward_rejected'] for s in stats])),
                'pairs': [p['id'] for p in pairs], 'step_time_s': round(time.time() - t0, 2),
            }
            history.append(rec)
            with STEP_LOG.open('a', encoding='utf-8') as f:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            print(f"step {rec['step']:>2}/{DPO['max_steps']}  loss={rec['loss']:.4f}  "
                  f"margin pol/ref={rec['policy_margin']:.3f}/{rec['ref_margin']:.3f}  "
                  f"reward_acc={rec['reward_accuracy']:.2f}  grad={grad_norm:.2f}  {rec['step_time_s']:.1f}s")
        status = 'completed'
    finally:
        DPO_TRAINED_IN_SESSION = len(history) > 0
        train_time = round(time.time() - t_start, 1)
        if status != 'completed':                                # частичный результат тоже сохраняется
            save_metrics(status, history, {'train_time_s': train_time})

    # --- Сохранение обучаемого LoRA в отдельный каталог ---
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    policy_model.save_pretrained(str(ADAPTER_DIR), selected_adapters=[POLICY])
    processor.save_pretrained(str(ADAPTER_DIR))

    # --- Проверки критериев ---
    assert len(history) == DPO['max_steps'], 'бюджет optimizer steps не выполнен'
    assert all(math.isfinite(h['loss']) for h in history), 'есть неконечный loss'
    saved = load_file(str(ADAPTER_DIR / 'adapter_model.safetensors'))
    saved = {k.removeprefix('base_model.model.'): v for k, v in saved.items()}
    cur = {k.removeprefix('base_model.model.'): p for k, p in policy_state().items()}
    assert set(saved) == set(cur), 'сохранены не все тензоры адаптера'
    saved_diff = max((saved[k].float() - cur[k].detach().float().cpu()).abs().max().item() for k in cur)
    assert saved_diff < 1e-6, f'сохранённый адаптер отличается от обученного: {saved_diff}'
    lab9_sd = load_file(str(LAB9_ADAPTER / 'adapter_model.safetensors')) if USE_LAB9 else None
    delta_vs_start = max((saved[k].float() - lab9_sd['base_model.model.' + k].float()).abs().max().item()
                         for k in saved) if USE_LAB9 else None
    if USE_LAB9:
        assert delta_vs_start > 0, 'веса политики не изменились — обучения не было'
    ref_unchanged = params_sha256(adapter_params(policy_model, REF)) == REF_SHA256
    assert ref_unchanged, 'reference изменился во время обучения'

    dpo_metrics = save_metrics('completed', history, {
        'train_time_s': train_time,
        'peak_gpu_memory_gb': round(torch.cuda.max_memory_allocated() / 2**30, 2),
        'adapter': {'dir': str(ADAPTER_DIR), 'adapter_name': POLICY,
                    'file_sha256': hashlib.sha256((ADAPTER_DIR / 'adapter_model.safetensors').read_bytes()).hexdigest(),
                    'policy_params_sha256': params_sha256(policy_state()),
                    'saved_vs_memory_max_abs_diff': saved_diff,
                    'max_abs_weight_change_vs_start': delta_vs_start},
        'reference_unchanged': ref_unchanged, 'reference_sha256': REF_SHA256,
        'step_log': str(STEP_LOG),
    })
    (ADAPTER_DIR / 'dpo_run_config.json').write_text(
        json.dumps({'protocol': dpo_protocol, 'metrics_file': str(METRICS_PATH)}, ensure_ascii=False, indent=2,
                   default=str), encoding='utf-8')

h = dpo_metrics['history']
print(f"\nСтатус: {dpo_metrics['status']} | {'ПОЛНЫЙ эксперимент' if dpo_metrics['is_full_experiment'] else 'SMOKE — не полный эксперимент'}")
print(f"Бюджет: {dpo_metrics['budget']['done_optimizer_steps']}/{dpo_metrics['budget']['planned_optimizer_steps']} "
      f"optimizer steps, {dpo_metrics['budget']['pairs_done']} пар; время {dpo_metrics.get('train_time_s')} с")
print(f"Loss: первый шаг {h[0]['loss']:.4f} -> последний {h[-1]['loss']:.4f} "
      f"(среднее первых 5: {dpo_metrics['loss_mean_first5']:.4f}, последних 5: {dpo_metrics['loss_mean_last5']:.4f}); "
      f"все конечны: {dpo_metrics['all_losses_finite']}")
print(f"Policy margin: {h[0]['policy_margin']:.3f} -> {h[-1]['policy_margin']:.3f} (ref margin на этих же парах: "
      f"{h[0]['ref_margin']:.3f} / {h[-1]['ref_margin']:.3f}); reward accuracy последнего шага {h[-1]['reward_accuracy']:.2f}")
print(f"Адаптер: {dpo_metrics['adapter']['dir']} | reference не изменился: {dpo_metrics['reference_unchanged']}")
print(f'Метрики: {METRICS_PATH}')

Загружен сохранённый завершённый запуск: artifacts\lab11\aligned_adapter\train  (FORCE_RETRAIN=True — переобучить)

Статус: completed | ПОЛНЫЙ эксперимент
Бюджет: 40/40 optimizer steps, 320 пар; время 1101.2 с
Loss: первый шаг 0.6931 -> последний 0.6078 (среднее первых 5: 0.6913, последних 5: 0.6158); все конечны: True
Policy margin: 5.328 -> 6.844 (ref margin на этих же парах: 5.328 / 5.047); reward accuracy последнего шага 1.00
Адаптер: artifacts\lab11\aligned_adapter\train | reference не изменился: True
Метрики: artifacts\lab11\dpo_metrics.json


Обучение модели

#### Что значат показатели

| Показатель | Что это | Как читать |
|---|---|---|
| **loss** | штраф, который обучение старается уменьшить | на старте 0,693, чем ниже, тем лучше |
| **policy margin** | насколько обучаемая модель предпочитает правильный ответ неправильному | чем больше, тем увереннее выбирает правильный |
| **ref margin** | то же самое у замороженной модели на тех же 8 парах | точка сравнения |
| **reward accuracy** | доля пар из 8, где после обучения правильный ответ усилился относительно неправильного сильнее, чем у reference | 1,00 значит, что во всех 8 парах |
| **grad** | норма градиента, то есть насколько сильно нужно менять веса на этом шаге | важно, чтобы не было резких скачков и NaN |
| **последнее число** | время шага в секундах | около 27 с |

**Как считается margin:**

$$\text{margin} = \log p(\text{правильный}) - \log p(\text{неправильный})$$

#### Что изменилось за обучение

| Шаг | loss | policy margin | ref margin | Разница | reward accuracy |
|---:|---:|---:|---:|---:|---:|
| 1 | 0,693 | 5,33 | 5,33 | 0 | 0,00 |
| 10 | 0,670 | 6,56 | 6,09 | 0,47 | 1,00 |
| 20 | 0,657 | 5,84 | 5,11 | 0,73 | 0,88 |
| 30 | 0,620 | 8,41 | 6,89 | 1,52 | 1,00 |
| 40 | 0,608 | 6,84 | 5,05 | 1,80 | 1,00 |

### Задание 7. Исходные ответы на тесте

*Вес: 1 балл.*

**Постановка.** Получите детерминированные ответы foundation и reference на одних отложенных изображениях. Если собственного SFT не было, прямо отметьте, что эти начальные политики совпадают.

**Результат.** Парные исходные ответы.

**Критерий принятия.** Не используется train; ответы декодируются без prompt; происхождение reference указано.


Получаем ответы двух моделей до DPO.<br>
`foundation` - исходная Qwen без адаптеров, твоего обучения в ней нет<br>
`reference` - Qwen + адаптер из лабы 9 (SFT)

Берутся 64 вопроса с картинками из id_test

**Задание 7. Исходные ответы на тесте**

**Что делали**: получили greedy-ответы foundation (Qwen без адаптеров) и reference (SFT работы 09) на 64 изображения id_test. Prompt отрезали, из ответа извлекли букву варианта.<br>
**Зачем**: зафиксировать ответы моделей до DPO, чтобы потом сравнить с ними aligned.<br>
**Результат**: train не используется (пересечений с ним нет). Ответы детерминированы: повторная генерация совпала. Одинаковых ответов у foundation и reference 50 из 64. У foundation 1 недопустимый ответ (`scienceqa-test-36`: «C» при двух вариантах). Файлы `baseline_predictions.jsonl`, `baseline_info.json`.

In [8]:
# Исходные ответы на тесте: foundation и reference на одних отложенных изображениях (ID test).
#   foundation — опубликованная Qwen2.5-VL-3B-Instruct без адаптеров (adapters disabled)
#   reference  — замороженный адаптер REF (SFT работы 09, если он был; иначе совпадает с foundation)
# Декодирование детерминированное (greedy), в ответ попадают только новые токены — prompt отрезается.
import re

GEN = dict(max_new_tokens=8, do_sample=False, num_beams=1)     # как в работах 09–10
FOUNDATION = 'foundation'
BASELINE_PATH = ARTIFACTS / 'baseline_predictions.jsonl'
BASELINE_INFO_PATH = ARTIFACTS / 'baseline_info.json'
FORCE_REGENERATE = False   # True — сгенерировать заново, даже если сохранённые ответы подходят

# --- Отложенная выборка: только ID test, train не используется ---
EVAL_PAIRS = list(test_pairs)
train_ids = {p['id'] for p in train_pairs}
train_hashes = {p['image_sha256'] for p in train_pairs}
assert all(p['split'] == 'id_test' for p in EVAL_PAIRS), 'в оценку попал не id_test'
assert not ({p['id'] for p in EVAL_PAIRS} & train_ids), 'пересечение с train по ID'
assert not ({p['image_sha256'] for p in EVAL_PAIRS} & train_hashes), 'пересечение с train по изображениям'

def build_generation_inputs(pair, image=None):
    """Тот же prompt, что при обучении и scoring: изображение, затем вопрос; prompt кончается репликой assistant."""
    image = load_image(pair['image']) if image is None else image
    prompt_text = processor.apply_chat_template(prompt_messages(pair), tokenize=False, add_generation_prompt=True)
    return processor(text=[prompt_text], images=[image], return_tensors='pt', add_special_tokens=False)

@torch.inference_mode()
def generate_answer(pair, policy, image=None):
    """Greedy-ответ политики policy ∈ {FOUNDATION, REF, POLICY}; возвращает только продолжение после prompt."""
    policy_model.eval()
    inputs = build_generation_inputs(pair, image).to(policy_model.device)
    kwargs = dict(**inputs, **GEN, use_cache=True, temperature=None, top_p=None, top_k=None,
                  pad_token_id=processor.tokenizer.pad_token_id)
    if policy == FOUNDATION:
        with policy_model.disable_adapter():
            out = policy_model.generate(**kwargs)
    else:
        policy_model.set_adapter(policy)
        try:
            out = policy_model.generate(**kwargs)
        finally:
            policy_model.set_adapter(POLICY); freeze_reference(policy_model)
    prompt_len = inputs['input_ids'].shape[1]
    new_tokens = out[0, prompt_len:]                        # декодируем без prompt
    assert torch.equal(out[0, :prompt_len], inputs['input_ids'][0]), 'generate изменил prompt'
    return processor.decode(new_tokens, skip_special_tokens=True).strip()

def parse_letter(text, n_choices):
    """Первая отдельная буква СУЩЕСТВУЮЩЕГО варианта (A, B, ...) в ответе; None, если её нет
    (например, «C» при двух вариантах — недопустимый ответ)."""
    letters = ''.join(chr(65 + i) for i in range(n_choices))
    m = re.search(rf'\b([{letters}])\b', text)
    return m.group(1) if m else None

# --- Кэш: сохранённые ответы используются, если совпадают ID, изображения, вопросы, reference и декодирование ---
assert params_sha256(adapter_params(policy_model, REF)) == REF_SHA256, 'reference изменён'

def load_baseline_cache():
    if FORCE_REGENERATE or not (BASELINE_PATH.is_file() and BASELINE_INFO_PATH.is_file()):
        return None
    info = json.loads(BASELINE_INFO_PATH.read_text(encoding='utf-8'))
    rows = [json.loads(s) for s in BASELINE_PATH.read_text(encoding='utf-8').splitlines() if s.strip()]
    by_id = {r['id']: r for r in rows}
    ok = (info.get('reference', {}).get('params_sha256') == REF_SHA256
          and all(info.get('decoding', {}).get(k) == v for k, v in GEN.items())
          and len(by_id) == len(rows) == len(EVAL_PAIRS)
          and all(p['id'] in by_id and by_id[p['id']]['image_sha256'] == p['image_sha256']
                  and by_id[p['id']]['question_sha256'] == hashlib.sha256(p['prompt'].encode('utf-8')).hexdigest()
                  for p in EVAL_PAIRS))
    return [by_id[p['id']] for p in EVAL_PAIRS] if ok else None

baseline_rows = load_baseline_cache()
BASELINE_FROM_CACHE = baseline_rows is not None
if BASELINE_FROM_CACHE:
    print(f'Сохранённые исходные ответы подходят — загружены из {BASELINE_PATH.name} (FORCE_REGENERATE=True — пересчитать)')

# --- Генерация (только если кэша нет) ---
torch.manual_seed(SEED)
t0 = time.time()
for i, pair in enumerate([] if BASELINE_FROM_CACHE else EVAL_PAIRS, 1):
    if i == 1:
        baseline_rows = []
    image = load_image(pair['image'])
    answers = {pol: generate_answer(pair, pol, image) for pol in (FOUNDATION, REF)}
    baseline_rows.append({
        'id': pair['id'], 'split': pair['split'], 'image_sha256': pair['image_sha256'],
        'question_sha256': hashlib.sha256(pair['prompt'].encode('utf-8')).hexdigest(),
        'gold': pair['chosen'], 'n_choices': len(pair['choices']),
        'foundation_raw': answers[FOUNDATION], 'reference_raw': answers[REF],
        'foundation_letter': parse_letter(answers[FOUNDATION], len(pair['choices'])),
        'reference_letter': parse_letter(answers[REF], len(pair['choices'])),
    })
    if i % 16 == 0 or i == len(EVAL_PAIRS):
        print(f'  {i}/{len(EVAL_PAIRS)} изображений, {time.time() - t0:.0f} с')

# Детерминизм (и соответствие кэша модели): повторная генерация на первых изображениях даёт те же ответы
for row, pair in zip(baseline_rows[:3], EVAL_PAIRS[:3]):
    assert generate_answer(pair, FOUNDATION) == row['foundation_raw'], 'foundation недетерминирован'
    assert generate_answer(pair, REF) == row['reference_raw'], 'reference недетерминирован'

BASELINE_PATH.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in baseline_rows), encoding='utf-8')

# --- Происхождение reference и сводка ---
same_raw = sum(r['foundation_raw'] == r['reference_raw'] for r in baseline_rows)
baseline_info = {
    'eval_split': 'id_test', 'n_images': len(baseline_rows), 'train_used': False,
    'decoding': {**GEN, 'deterministic': True, 'prompt_stripped': True},
    'foundation': {'model': MODEL_ID, 'adapters': 'disabled', 'quantization': '4-bit nf4 (как при обучении)'},
    'reference': {'adapter_name': REF, 'source': SOURCE, 'continuation_of_own_sft': USE_LAB9,
                  'origin': (f'SFT-адаптер работы 09: {LAB9_ADAPTER}' if USE_LAB9 else
                             'собственного SFT не было: reference = foundation (новый LoRA с B = 0), '
                             'начальные политики СОВПАДАЮТ'),
                  'params_sha256': REF_SHA256},
    'policies_identical_by_construction': not USE_LAB9,
    'identical_raw_answers': same_raw,
    'unparsed_letters': {'foundation': sum(r['foundation_letter'] is None for r in baseline_rows),
                         'reference': sum(r['reference_letter'] is None for r in baseline_rows)},
    'file': str(BASELINE_PATH),
    'loaded_from_cache': BASELINE_FROM_CACHE,
}
save_json('baseline_info.json', baseline_info)

print(f"\nОтложенная выборка: id_test, {len(baseline_rows)} изображений (train не используется)")
print(f"Foundation: {MODEL_ID} без адаптеров | Reference: {baseline_info['reference']['origin']}")
if not USE_LAB9:
    print('ВНИМАНИЕ: собственного SFT не было — foundation и reference являются одной и той же начальной политикой.')
print(f"Одинаковых сырых ответов foundation/reference: {same_raw} из {len(baseline_rows)}; "
      f"без допустимой буквы варианта: foundation {baseline_info['unparsed_letters']['foundation']}, "
      f"reference {baseline_info['unparsed_letters']['reference']}")
print('\nПримеры (ответы декодированы без prompt):')
print(f"  {'id':<22} {'верно':^5} {'foundation':<14} {'reference':<14}")
for r in baseline_rows[:8]:
    print(f"  {r['id']:<22} {r['gold']:^5} {r['foundation_raw'][:14]!r:<14} {r['reference_raw'][:14]!r:<14}")
print(f'-> {BASELINE_PATH}')

Сохранённые исходные ответы подходят — загружены из baseline_predictions.jsonl (FORCE_REGENERATE=True — пересчитать)

Отложенная выборка: id_test, 64 изображений (train не используется)
Foundation: Qwen/Qwen2.5-VL-3B-Instruct без адаптеров | Reference: SFT-адаптер работы 09: artifacts\lab9\lab9_adapter
Одинаковых сырых ответов foundation/reference: 50 из 64; без допустимой буквы варианта: foundation 1, reference 0

Примеры (ответы декодированы без prompt):
  id                     верно foundation     reference     
  scienceqa-test-10        C   'C'            'C'           
  scienceqa-test-17        B   'B'            'B'           
  scienceqa-test-18        A   'A'            'A'           
  scienceqa-test-35        B   'B'            'B'           
  scienceqa-test-36        B   'C'            'B'           
  scienceqa-test-39        A   'A'            'A'           
  scienceqa-test-42        B   'B'            'B'           
  scienceqa-test-45        A   'A'            'A'  

### Задание 8. Ответы после DPO

*Вес: 1 балл.*

**Постановка.** Активируйте обученный адаптер и получите ответы для тех же ID. Сохраните foundation/reference/aligned вместе с правильной буквой.

**Результат.** blind_predictions.jsonl.

**Критерий принятия.** Каждый ID имеет три фактических ответа; порядок выборок не используется вместо ID.


Получаем ответы модели обученной методом DPO

**Задание 8. Ответы после DPO**

**Что делали**: включили обученный адаптер (сверили его sha256), получили ответы aligned на те же 64 ID и свели три политики по ID вместе с правильной буквой.<br>
**Зачем**: получить ответы модели после DPO и подготовить слепую форму, где названия моделей скрыты.<br>
**Результат**: у каждого из 64 ID три ответа. DPO изменил 1 ответ reference (`scienceqa-test-349`: B → C, исправление). Foundation и aligned различаются в 15 ответах, foundation и reference — в 14. Файл `blind_predictions.jsonl` (со слепыми слотами answer_1/2/3).

In [9]:
# Ответы после DPO: активируем обученный адаптер POLICY ('default') и генерируем для тех же ID.
# Все три политики сводятся по ID (не по порядку строк) в blind_predictions.jsonl вместе с правильной буквой.
ALIGNED = 'aligned'
POLICIES = (FOUNDATION, 'reference', ALIGNED)
BLIND_PATH = ARTIFACTS / 'blind_predictions.jsonl'
ALIGNED_DIR = globals().get('ADAPTER_DIR', ARTIFACTS / 'aligned_adapter' / ('smoke' if FAST_DEV_RUN else 'train'))
ALIGNED_METRICS = ARTIFACTS / ('dpo_metrics_smoke.json' if FAST_DEV_RUN else 'dpo_metrics.json')
FORCE_REGENERATE_ALIGNED = False   # True — сгенерировать ответы aligned заново

# --- 1. Активируем именно обученный адаптер (сверка по sha256 из dpo_metrics) ---
trained = json.loads(ALIGNED_METRICS.read_text(encoding='utf-8'))
assert trained['status'] == 'completed', f"DPO не завершено: {trained['status']}"
expected_sha = trained['adapter']['policy_params_sha256']
if params_sha256(adapter_params(policy_model, POLICY)) != expected_sha:
    # ядро перезапускалось — загружаем сохранённые веса обученного LoRA в адаптер POLICY
    saved = {k.removeprefix('base_model.model.'): v
             for k, v in load_file(str(ALIGNED_DIR / 'adapter_model.safetensors')).items()}
    cur = {k.removeprefix('base_model.model.'): p for k, p in adapter_params(policy_model, POLICY).items()}
    assert set(saved) == set(cur), 'набор тензоров сохранённого адаптера не совпадает'
    with torch.no_grad():
        for k, p in cur.items():
            p.copy_(saved[k].to(p.device, p.dtype))
    print(f'Обученный адаптер загружен из {ALIGNED_DIR}')
assert params_sha256(adapter_params(policy_model, POLICY)) == expected_sha, 'активен не обученный адаптер'
assert params_sha256(adapter_params(policy_model, REF)) == REF_SHA256, 'reference изменён'
assert params_sha256(adapter_params(policy_model, POLICY)) != REF_SHA256, 'aligned совпадает с reference — DPO не применён'
policy_model.set_adapter(POLICY); freeze_reference(policy_model)

# --- 2. Исходные ответы — по ID ---
baseline_by_id = {}
for line in BASELINE_PATH.read_text(encoding='utf-8').splitlines():
    r = json.loads(line)
    assert r['id'] not in baseline_by_id, f"дубль ID в baseline: {r['id']}"
    baseline_by_id[r['id']] = r
eval_by_id = {p['id']: p for p in EVAL_PAIRS}
assert set(baseline_by_id) == set(eval_by_id), 'ID исходных ответов не совпадают с отложенной выборкой'

# --- 3. Ответы aligned: кэш, если сохранены для ЭТОГО адаптера и тех же ID; иначе генерация ---
aligned_by_id = {}
if BLIND_PATH.is_file() and not FORCE_REGENERATE_ALIGNED:
    cached = {}
    for s in BLIND_PATH.read_text(encoding='utf-8').splitlines():
        if s.strip():
            r = json.loads(s); cached[r['id']] = r
    if (set(cached) == set(eval_by_id)
            and all(r.get('aligned_adapter_sha256') == expected_sha
                    and r['image_sha256'] == eval_by_id[i]['image_sha256'] for i, r in cached.items())):
        aligned_by_id = {i: r['answers'][ALIGNED] for i, r in cached.items()}
        print(f'Сохранённые ответы aligned подходят — загружены из {BLIND_PATH.name} '
              f'(FORCE_REGENERATE_ALIGNED=True — пересчитать)')
ALIGNED_FROM_CACHE = bool(aligned_by_id)

# Генерация: обходим ID в перемешанном порядке и сохраняем по ID
gen_order = sorted(eval_by_id)
random.Random(SEED + 1).shuffle(gen_order)          # порядок генерации ≠ порядку baseline
torch.manual_seed(SEED)
t0 = time.time()
for i, pid in enumerate([] if ALIGNED_FROM_CACHE else gen_order, 1):
    aligned_by_id[pid] = generate_answer(eval_by_id[pid], POLICY)
    if i % 16 == 0 or i == len(gen_order):
        print(f'  aligned: {i}/{len(gen_order)}, {time.time() - t0:.0f} с')

# Детерминизм aligned и неизменность исходных политик на нескольких ID
for pid in sorted(eval_by_id)[:3]:
    assert generate_answer(eval_by_id[pid], POLICY) == aligned_by_id[pid], 'aligned недетерминирован'
    assert generate_answer(eval_by_id[pid], REF) == baseline_by_id[pid]['reference_raw'], 'reference дал другой ответ'
    assert generate_answer(eval_by_id[pid], FOUNDATION) == baseline_by_id[pid]['foundation_raw'], 'foundation дал другой ответ'

# --- 4. Сводка по ID ---
def blind_order(pid):
    """Фиксированная перестановка политик для слепой проверки (зависит от ID и SEED, не от позиции)."""
    order = list(POLICIES)
    random.Random(int(hashlib.sha256(f'{SEED}:{pid}'.encode()).hexdigest(), 16)).shuffle(order)
    return order

blind_rows = []
for pid in sorted(eval_by_id):                       # сортировка по ID — только для стабильного файла
    pair, base = eval_by_id[pid], baseline_by_id[pid]
    assert base['image_sha256'] == pair['image_sha256'], f'{pid}: другое изображение'
    assert base['question_sha256'] == hashlib.sha256(pair['prompt'].encode('utf-8')).hexdigest(), f'{pid}: другой вопрос'
    raw = {FOUNDATION: base['foundation_raw'], 'reference': base['reference_raw'], ALIGNED: aligned_by_id[pid]}
    order = blind_order(pid)
    blind_rows.append({
        'id': pid, 'split': pair['split'], 'image': pair['image'], 'image_sha256': pair['image_sha256'],
        'question': pair['prompt'], 'choices': pair['choices'],
        'gold': pair['chosen'], 'gold_text': pair['chosen_text'],
        'answers': raw,
        'letters': {pol: parse_letter(raw[pol], len(pair['choices'])) for pol in POLICIES},
        'blind_slots': {f'answer_{i + 1}': pol for i, pol in enumerate(order)},   # ключ для слепой формы
        'reference_source': SOURCE,
        'aligned_adapter_sha256': expected_sha,
    })

# --- Критерии: у каждого ID три фактических ответа; сведение по ID ---
assert len(blind_rows) == len(EVAL_PAIRS) and len({r['id'] for r in blind_rows}) == len(blind_rows)
for r in blind_rows:
    assert set(r['answers']) == set(POLICIES), r['id']
    assert all(isinstance(r['answers'][p], str) for p in POLICIES), f"{r['id']}: нет ответа"
    assert sorted(r['blind_slots'].values()) == sorted(POLICIES)

BLIND_PATH.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in blind_rows), encoding='utf-8')

n = len(blind_rows)
diff = lambda a, b: sum(r['answers'][a] != r['answers'][b] for r in blind_rows)
print(f"\nblind_predictions.jsonl: {n} ID × 3 политики (foundation / reference / aligned) + правильная буква")
print(f"Обученный адаптер: {ALIGNED_DIR} (sha256 сверен) | reference: {SOURCE}")
print(f"Различающихся ответов: reference vs aligned — {diff('reference', ALIGNED)}, "
      f"foundation vs aligned — {diff(FOUNDATION, ALIGNED)}, foundation vs reference — {diff(FOUNDATION, 'reference')}")
changed = [r for r in blind_rows if r['answers']['reference'] != r['answers'][ALIGNED]]
print(f"\nID, где DPO изменил ответ reference ({len(changed)}):" if changed else '\nDPO не изменил ни одного ответа reference.')
print(f"  {'id':<22} {'верно':^5} {'foundation':<12} {'reference':<12} {'aligned':<12}")
for r in (changed or blind_rows)[:10]:
    a = r['answers']
    print(f"  {r['id']:<22} {r['gold']:^5} {a[FOUNDATION][:12]!r:<12} {a['reference'][:12]!r:<12} {a[ALIGNED][:12]!r:<12}")
print(f'-> {BLIND_PATH}')

Сохранённые ответы aligned подходят — загружены из blind_predictions.jsonl (FORCE_REGENERATE_ALIGNED=True — пересчитать)

blind_predictions.jsonl: 64 ID × 3 политики (foundation / reference / aligned) + правильная буква
Обученный адаптер: artifacts\lab11\aligned_adapter\train (sha256 сверен) | reference: lab09_sft_adapter
Различающихся ответов: reference vs aligned — 1, foundation vs aligned — 15, foundation vs reference — 14

ID, где DPO изменил ответ reference (1):
  id                     верно foundation   reference    aligned     
  scienceqa-test-349       C   'B'          'B'          'C'         
-> artifacts\lab11\blind_predictions.jsonl


### Задание 9. Автоматическая и ручная оценка

*Вес: 1 балл.*

**Постановка.** Рассчитайте EM по эталонным буквам. Создайте форму ручной проверки предпочтений, галлюцинаций и формата; заполните её самостоятельно после просмотра ответов. Пропуски обозначайте pending, а не нулём.

**Результат.** objective_scores.json и blind_judgements.jsonl.

**Критерий принятия.** EM не назван человеческим win rate; неопределённые суждения не включены в ручные метрики; для финальной сдачи форма заполнена.


Оцениваем ответы трех моделей вручную

`foundation` - только Qwen, адаптеры выключены<br>
`reference` - Qwen + адаптер лабы 9<br>
`aligned` - Qwen + адаптер лабы 9, дообученный методом DPO

**Задание 9. Автоматическая и ручная оценка**

**Что делали**: посчитали EM по эталонным буквам и заполнили слепую форму: лучший ответ, галлюцинация и формат для каждой из 64 карточек.<br>
**Зачем**: оценить ответы двумя способами: автоматически (совпала ли буква) и вручную (качество ответа глазами человека).<br>
**Результат**: EM — foundation 55/64 = 85,9 %, reference 57/64 = 89,1 %, aligned 58/64 = 90,6 %. Только буква в ответе: 89,1 % / 100 % / 100 %. aligned − reference: +1,6 п.п., McNemar p = 1,0. Форма заполнена: 64 / 64, без pending и unsure. Файлы `objective_scores.json`, `blind_judgements.jsonl`.

In [10]:
# Автоматическая и ручная оценка.
# 1) EM (exact match) по эталонным буквам -> objective_scores.json. EM — автоматическая метрика, НЕ человеческий win rate.
# 2) Форма слепой ручной проверки -> blind_judgements.jsonl (по записи на каждый ID): ответы показаны как
#    answer_1/2/3 без названий политик; ключ слотов хранится отдельно в blind_predictions.jsonl.
#    Пропуски = 'pending' (не 0); 'unsure' — неопределённое суждение. Оба в ручные метрики не входят.
import io, datetime
from collections import Counter
from scipy.stats import binomtest
import ipywidgets as W
from IPython.display import display

OBJECTIVE_PATH = ARTIFACTS / 'objective_scores.json'
JUDGE_PATH = ARTIFACTS / 'blind_judgements.jsonl'
REQUIRE_COMPLETE_FORM = False   # True перед финальной сдачей: ячейка упадёт, если остались pending
SLOTS = ('answer_1', 'answer_2', 'answer_3')
PREF_VALUES = ('pending', 'answer_1', 'answer_2', 'answer_3', 'tie', 'unsure')
HALL_VALUES = ('pending', 'no', 'yes', 'unsure')
FMT_VALUES = ('pending', 'ok', 'bad', 'unsure')
EXCLUDED = ('pending', 'unsure')   # не входят в ручные метрики
# Подсказка по умолчанию: подставляется в виджет при ОТКРЫТИИ записи и сохраняется только после просмотра
# (кнопка или переход). Неоткрытые записи остаются pending.
FORM_DEFAULTS = {'preference': 'tie', 'hallucination': 'no', 'format': 'ok'}

blind = [json.loads(s) for s in BLIND_PATH.read_text(encoding='utf-8').splitlines() if s.strip()]
blind_by_id = {r['id']: r for r in blind}

# ---------------- 1. Автоматическая метрика: EM по эталонным буквам ----------------
def wilson(k, n, z=1.96):
    if n == 0: return [None, None]
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return [round(c - h, 4), round(c + h, 4)]

def strict_format(text, n_choices):
    """Автоматическая проверка формата: ответ — ровно одна буква существующего варианта (как просит prompt)."""
    return re.fullmatch(rf"[{''.join(chr(65 + i) for i in range(n_choices))}]", text.strip()) is not None

n_eval = len(blind)
em = {}
for pol in POLICIES:
    correct = [r['letters'][pol] == r['gold'] for r in blind]
    k = sum(correct)
    em[pol] = {'em': round(k / n_eval, 4), 'correct': k, 'n': n_eval, 'ci95_wilson': wilson(k, n_eval),
               'no_valid_letter': sum(r['letters'][pol] is None for r in blind),
               'strict_letter_format': round(np.mean([strict_format(r['answers'][pol], len(r['choices'])) for r in blind]), 4)}

def paired(a, b):
    """Парное сравнение EM на одних ID; точный тест McNemar по расходящимся парам."""
    ca = {r['id']: r['letters'][a] == r['gold'] for r in blind}
    cb = {r['id']: r['letters'][b] == r['gold'] for r in blind}
    only_a = sum(ca[i] and not cb[i] for i in ca); only_b = sum(cb[i] and not ca[i] for i in ca)
    p = binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
    return {'both_correct': sum(ca[i] and cb[i] for i in ca), 'both_wrong': sum(not ca[i] and not cb[i] for i in ca),
            f'only_{a}': only_a, f'only_{b}': only_b, 'delta_em': round((only_a - only_b) / n_eval, 4),
            'mcnemar_exact_p': round(p, 4)}

objective_scores = {
    'metric': 'exact_match',
    'definition': 'доля ID, где первая допустимая буква ответа совпадает с эталонной буквой ScienceQA',
    'is_human_win_rate': False,
    'note': 'EM — автоматическая метрика по эталонной разметке. Это НЕ человеческий win rate и не оценка полезности; '
            'win rate считается только по заполненной слепой форме.',
    'eval_split': 'id_test', 'n': n_eval,
    'per_policy': em,
    'paired': {'aligned_vs_reference': paired(ALIGNED, 'reference'),
               'reference_vs_foundation': paired('reference', FOUNDATION),
               'aligned_vs_foundation': paired(ALIGNED, FOUNDATION)},
    'reference_source': SOURCE,
}
save_json(OBJECTIVE_PATH.name, objective_scores)

print(f'EM по эталонным буквам (id_test, n = {n_eval}) — автоматическая метрика, не human win rate:')
print(f"  {'политика':<12}{'EM':>8}{'верно':>8}   {'95% CI (Wilson)':<18}{'нет буквы':>10}{'строгий формат':>16}")
for pol in POLICIES:
    e = em[pol]
    print(f"  {pol:<12}{e['em']:>8.1%}{e['correct']:>5}/{e['n']:<3}  [{e['ci95_wilson'][0]:.3f}, {e['ci95_wilson'][1]:.3f}]"
          f"{e['no_valid_letter']:>10}{e['strict_letter_format']:>15.1%}")
pv = objective_scores['paired']['aligned_vs_reference']
print(f"  aligned vs reference: только aligned верно — {pv['only_aligned']}, только reference — {pv['only_reference']}, "
      f"ΔEM = {pv['delta_em']:+.1%}, McNemar p = {pv['mcnemar_exact_p']}")

# ---------------- 2. Форма слепой ручной проверки ----------------
def empty_judgement():
    return {'preference': 'pending',
            'hallucination': {s: 'pending' for s in SLOTS},
            'format': {s: 'pending' for s in SLOTS},
            'notes': ''}

def form_record(r):
    """Запись формы: без названий политик, ответы в фиксированной слепой перестановке."""
    return {'id': r['id'], 'image': r['image'], 'question': r['question'], 'choices': r['choices'],
            'gold': r['gold'], 'gold_text': r['gold_text'],
            'answers': {slot: r['answers'][pol] for slot, pol in r['blind_slots'].items()},
            'judgement': empty_judgement(), 'judged_at': None}

def validate(rec):
    j = rec['judgement']
    assert j['preference'] in PREF_VALUES, (rec['id'], j['preference'])
    for s in SLOTS:
        assert j['hallucination'][s] in HALL_VALUES and j['format'][s] in FMT_VALUES, rec['id']
    assert isinstance(rec['judgement'].get('notes', ''), str)

def is_complete(rec):
    j = rec['judgement']
    return 'pending' not in [j['preference'], *j['hallucination'].values(), *j['format'].values()]

def save_judgements(recs):
    JUDGE_PATH.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in recs), encoding='utf-8')

# Уже заполненные суждения сохраняются; запись сбрасывается в pending, только если ответы изменились
existing = {}
if JUDGE_PATH.is_file():
    for s in JUDGE_PATH.read_text(encoding='utf-8').splitlines():
        if s.strip():
            r = json.loads(s); existing[r['id']] = r
judgements, reset_ids = [], []
for r in sorted(blind, key=lambda x: x['id']):
    rec = form_record(r)
    old = existing.get(r['id'])
    if old is not None:
        if old['answers'] == rec['answers']:
            rec['judgement'], rec['judged_at'] = old['judgement'], old.get('judged_at')
        else:
            reset_ids.append(r['id'])
    validate(rec)
    judgements.append(rec)
assert [r['id'] for r in judgements] == sorted(blind_by_id), 'форма должна содержать каждый ID ровно один раз'
save_judgements(judgements)
if reset_ids:
    print(f'\nВНИМАНИЕ: ответы изменились, суждения сброшены в pending для {len(reset_ids)} ID: {reset_ids[:5]}…')

GUIDE = """<b>Как заполнять (слепо: какой слот какой модели — не показано)</b><br>
<b>Лучший</b> — какой ответ лучше по правильности и соблюдению инструкции; <i>tie</i> — одинаково хороши/плохи
(в т.ч. если ответы идентичны); <i>unsure</i> — не можете решить.<br>
<b>Галлюцинация</b> — <i>yes</i>, если ответ называет несуществующий вариант (например «C» при двух вариантах)
или добавляет выдуманное содержание; неверная, но существующая буква — ошибка, а не галлюцинация.<br>
<b>Формат</b> — <i>ok</i>, если ответ — только буква варианта, как просит вопрос; <i>bad</i> — лишний текст/символы.<br>
Открытая запись предзаполнена: <i>tie / no / ok</i> — проверьте и исправьте, где нужно.<br>
Сохранение — кнопкой или при переходе; неоткрытые записи остаются <i>pending</i> (не ноль)."""

def judgement_form():
    state = {'i': next((k for k, r in enumerate(judgements) if not is_complete(r)), 0)}
    img = W.Image(format='png')
    info = W.HTML()
    pref = W.Dropdown(options=PREF_VALUES, description='Лучший:')
    hall = {s: W.Dropdown(options=HALL_VALUES, description=f'Галлюц. {s[-1]}:') for s in SLOTS}
    fmt = {s: W.Dropdown(options=FMT_VALUES, description=f'Формат {s[-1]}:') for s in SLOTS}
    notes = W.Text(description='Заметка:', layout=W.Layout(width='90%'))
    status = W.HTML()
    b_prev, b_next = W.Button(description='◀ Назад'), W.Button(description='Вперёд ▶')
    b_pending, b_save = W.Button(description='Следующий pending'), W.Button(description='Сохранить', button_style='success')

    def render():
        r = judgements[state['i']]
        im = Image.open(r['image']).convert('RGB'); im.thumbnail((420, 420))
        buf = io.BytesIO(); im.save(buf, format='PNG'); img.value = buf.getvalue()
        choices = '<br>'.join(f'{chr(65 + k)}. {c}' for k, c in enumerate(r['choices']))
        same = len(set(r['answers'].values())) == 1
        answers = ''.join(f"<tr><td><b>{s}</b></td><td><code>{r['answers'][s]!r}</code></td></tr>" for s in SLOTS)
        info.value = (f"<b>{state['i'] + 1}/{len(judgements)}</b> · {r['id']}<br>"
                      f"<p>{r['question'].split(chr(10))[0]}</p><p>{choices}</p>"
                      f"<p>Эталон: <b>{r['gold']}</b> ({r['gold_text']})</p><table>{answers}</table>"
                      + ('<p><i>Все три ответа идентичны.</i></p>' if same else ''))
        j = r['judgement']
        fill = lambda v, key: FORM_DEFAULTS[key] if v == 'pending' else v
        pref.value = fill(j['preference'], 'preference'); notes.value = j.get('notes', '')
        for s in SLOTS:
            hall[s].value = fill(j['hallucination'][s], 'hallucination')
            fmt[s].value = fill(j['format'][s], 'format')
        n_done = sum(is_complete(x) for x in judgements)
        hint = '' if is_complete(r) else ' · <i>эта запись предзаполнена, ещё не сохранена</i>'
        status.value = f'Заполнено полностью: <b>{n_done}/{len(judgements)}</b> · файл: {JUDGE_PATH.name}{hint}'

    def store(_=None):
        r = judgements[state['i']]
        r['judgement'] = {'preference': pref.value, 'hallucination': {s: hall[s].value for s in SLOTS},
                          'format': {s: fmt[s].value for s in SLOTS}, 'notes': notes.value}
        r['judged_at'] = datetime.datetime.now().isoformat(timespec='seconds') if is_complete(r) else None
        validate(r); save_judgements(judgements); render()

    def go(step):
        store(); state['i'] = (state['i'] + step) % len(judgements); render()

    def next_pending(_):
        store()
        order = list(range(state['i'] + 1, len(judgements))) + list(range(0, state['i'] + 1))
        state['i'] = next((k for k in order if not is_complete(judgements[k])), state['i']); render()

    b_prev.on_click(lambda _: go(-1)); b_next.on_click(lambda _: go(1))
    b_pending.on_click(next_pending); b_save.on_click(store)
    render()
    return W.VBox([W.HTML(GUIDE), W.HBox([img, info]), pref,
                   W.HBox([W.VBox([hall[s] for s in SLOTS]), W.VBox([fmt[s] for s in SLOTS])]),
                   notes, W.HBox([b_prev, b_next, b_pending, b_save]), status])

display(judgement_form())

# ---------------- Статус формы ----------------
def judgement_status(recs):
    """Сколько суждений реально заполнено; pending и unsure считаются отдельно и не входят в метрики."""
    pref = Counter(r['judgement']['preference'] for r in recs)
    hall = Counter(v for r in recs for v in r['judgement']['hallucination'].values())
    fmt = Counter(v for r in recs for v in r['judgement']['format'].values())
    return {'n_records': len(recs), 'n_complete': sum(is_complete(r) for r in recs),
            'preference': dict(pref), 'hallucination': dict(hall), 'format': dict(fmt),
            'usable_preference': sum(v for k, v in pref.items() if k not in EXCLUDED),
            'usable_hallucination': sum(v for k, v in hall.items() if k not in EXCLUDED),
            'usable_format': sum(v for k, v in fmt.items() if k not in EXCLUDED)}

form_status = judgement_status(judgements)
print(f"\nФорма {JUDGE_PATH}: записей {form_status['n_records']} (каждый ID), заполнено полностью {form_status['n_complete']}")
print(f"  годных суждений (без pending/unsure): предпочтение {form_status['usable_preference']}, "
      f"галлюцинации {form_status['usable_hallucination']}, формат {form_status['usable_format']}")
if form_status['n_complete'] < form_status['n_records']:
    print('  Форма ещё не заполнена: заполните её в виджете выше. Для финальной сдачи поставьте REQUIRE_COMPLETE_FORM = True.')
if REQUIRE_COMPLETE_FORM:
    assert form_status['n_complete'] == form_status['n_records'], \
        f"для финальной сдачи форма должна быть заполнена: pending осталось у {form_status['n_records'] - form_status['n_complete']} ID"

EM по эталонным буквам (id_test, n = 64) — автоматическая метрика, не human win rate:
  политика          EM   верно   95% CI (Wilson)    нет буквы  строгий формат
  foundation     85.9%   55/64   [0.754, 0.924]         1          89.1%
  reference      89.1%   57/64   [0.791, 0.946]         0         100.0%
  aligned        90.6%   58/64   [0.810, 0.956]         0         100.0%
  aligned vs reference: только aligned верно — 1, только reference — 0, ΔEM = +1.6%, McNemar p = 1.0



Форма artifacts\lab11\blind_judgements.jsonl: записей 64 (каждый ID), заполнено полностью 64
  годных суждений (без pending/unsure): предпочтение 64, галлюцинации 192, формат 192


### Задание 10. Эффект DPO

*Вес: 1 балл.*

**Постановка.** Сопоставьте EM трёх политик, а по заполненным суждениям — win rate, hallucination rate и соблюдение формата. Укажите число оценённых записей и ограничения label-derived предпочтений.

**Результат.** blind_report.json и вывод.

**Критерий принятия.** Метрики имеют реальные знаменатели; pending-оценка не выдаётся за завершённую; вывод не обобщает предпочтение правильной буквы на безопасность или универсальную полезность.


Дополнительный анализ ручных метрик

**Задание 10. Эффект DPO**

**Что делали**: раскрыли слепые слоты и посчитали по моделям EM, win rate, долю галлюцинаций и соблюдение формата. Указали число оценённых записей и ограничения label-derived предпочтений.<br>
**Зачем**: ответить, что дал DPO, без обобщений за пределы выбора правильной буквы.<br>
**Результат**: оценено 64 записи, ничья в 59. Лучшим выбран foundation 4 раза, reference 0, aligned 1. Галлюцинации 1 / 0 / 0 из 64, формат ok 59 / 64 / 64. DPO изменил 1 ответ из 64, прирост EM +1,6 п.п. не значим (p = 1,0). Win rate занижает поражения foundation: в 6 ничьих верны два ответа из трёх. Файл `blind_report.json`.

In [11]:
# Эффект DPO: EM трёх политик (автоматически) + по заполненным слепым суждениям win rate, hallucination rate
# и соблюдение формата. Слоты answer_1/2/3 раскрываются через blind_slots из blind_predictions.jsonl.
# Знаменатели — только реально вынесенные суждения: pending и unsure исключены и показаны отдельно.
REPORT_PATH = ARTIFACTS / 'blind_report.json'

blind_by_id = {r['id']: r for r in (json.loads(s) for s in BLIND_PATH.read_text(encoding='utf-8').splitlines() if s.strip())}
judged = [json.loads(s) for s in JUDGE_PATH.read_text(encoding='utf-8').splitlines() if s.strip()]
objective = json.loads(OBJECTIVE_PATH.read_text(encoding='utf-8'))
assert {r['id'] for r in judged} == set(blind_by_id), 'форма и предсказания содержат разные ID'
for r in judged:
    validate(r)
    b = blind_by_id[r['id']]
    assert {s: b['answers'][pol] for s, pol in b['blind_slots'].items()} == r['answers'], f"{r['id']}: ответы формы ≠ предсказаниям"

n_records = len(judged)
n_complete = sum(is_complete(r) for r in judged)
FORM_STATUS = 'complete' if n_complete == n_records else 'partial'

def rate(k, n):
    return {'k': k, 'n': n, 'rate': round(k / n, 4) if n else None, 'ci95_wilson': wilson(k, n)}

# --- Предпочтения: лучший слот -> политика ---
pref_rows = [(r, blind_by_id[r['id']]['blind_slots']) for r in judged if r['judgement']['preference'] not in EXCLUDED]
n_pref = len(pref_rows)
best_policy = [slots[r['judgement']['preference']] if r['judgement']['preference'] in slots else 'tie'
               for r, slots in pref_rows]
pref_counts = Counter(best_policy)
win_rate = {pol: {**rate(pref_counts[pol], n_pref),
                  'wins_excluding_ties': rate(pref_counts[pol], n_pref - pref_counts['tie'])} for pol in POLICIES}

def pairwise(a, b):
    """a против b по выбору лучшего из трёх: победа a — лучшим выбран a; поражение — выбран b; иначе не решено."""
    w, l = pref_counts[a], pref_counts[b]
    return {'wins': w, 'losses': l, 'undecided': n_pref - w - l, 'n_judged': n_pref,
            'win_rate_decided': round(w / (w + l), 4) if w + l else None,
            'sign_test_p': round(binomtest(w, w + l, 0.5).pvalue, 4) if w + l else None}

# --- Галлюцинации и формат по слотам -> политикам ---
def slot_metric(field, positive):
    out = {}
    for pol in POLICIES:
        vals = [r['judgement'][field][slot] for r in judged
                for slot, p in blind_by_id[r['id']]['blind_slots'].items() if p == pol]
        usable = [v for v in vals if v not in EXCLUDED]
        out[pol] = {**rate(sum(v == positive for v in usable), len(usable)),
                    'pending': vals.count('pending'), 'unsure': vals.count('unsure')}
    return out

hallucination = slot_metric('hallucination', 'yes')
format_ok = slot_metric('format', 'ok')

# Согласованность ручного формата с автоматической проверкой «ровно одна буква»
format_disagree = []
for r in judged:
    b = blind_by_id[r['id']]
    for slot, pol in b['blind_slots'].items():
        human = r['judgement']['format'][slot]
        if human in EXCLUDED: continue
        auto = 'ok' if strict_format(b['answers'][pol], len(b['choices'])) else 'bad'
        if human != auto:
            format_disagree.append({'id': r['id'], 'slot': slot, 'answer': b['answers'][pol], 'human': human, 'auto': auto})

# Перекос формы «один лучший или tie»: два верных ответа против одного неверного записываются как tie,
# и проигрыш неверной политики в win rate не попадает. Считаем такие записи по эталонным буквам.
tie_two_of_three = []
for r in judged:
    if r['judgement']['preference'] != 'tie': continue
    b = blind_by_id[r['id']]
    correct = [pol for pol in POLICIES if b['letters'][pol] == b['gold']]
    if len(correct) == 2:
        tie_two_of_three.append({'id': r['id'], 'correct': correct,
                                 'wrong': next(p for p in POLICIES if p not in correct)})
tie_loser_counts = Counter(t['wrong'] for t in tie_two_of_three)

# --- Что изменил DPO ---
dpo_changed = [i for i, b in blind_by_id.items() if b['answers']['reference'] != b['answers'][ALIGNED]]
pv = objective['paired']['aligned_vs_reference']

LIMITATIONS = [
    'Предпочтения для DPO выведены из эталонной разметки ScienceQA (chosen = верная буква, rejected = следующий '
    'неверный вариант), а не из экспертных оценок полезности: обучение оптимизировало выбор правильной буквы.',
    'Вывод относится только к ответам-буквам на вопросы ScienceQA (natural science, 1–6 классы, id_test, n = 64); '
    'он не переносится на безопасность, развёрнутые ответы, другие предметы или универсальную полезность.',
    'Ручные суждения вынес один оценщик (студент); слоты скрыты, но эталонная буква была видна оценщику.',
    'Форма выбирает один лучший ответ или tie: если два ответа одинаково верны, а третий хуже, это фиксируется как '
    'tie (с заметкой), поэтому win rate занижает различия между политиками.',
    'Ответы — одна буква: «галлюцинация» сводится к несуществующему варианту, «формат» — к лишним символам.',
    'Малая выборка: различия EM в 1–2 ответа статистически не значимы (см. McNemar).',
]

blind_report = {
    'eval_split': 'id_test', 'n_records': n_records,
    'form_status': FORM_STATUS, 'n_complete': n_complete, 'n_pending_records': n_records - n_complete,
    'manual_metrics_status': 'final' if FORM_STATUS == 'complete' else
                             'ПРЕДВАРИТЕЛЬНО: форма заполнена не полностью, метрики по части записей',
    'objective': {
        'metric': 'exact_match по эталонным буквам (автоматическая, НЕ human win rate)',
        'per_policy': {pol: objective['per_policy'][pol] for pol in POLICIES},
        'paired': objective['paired'],
    },
    'manual': {
        'preference': {'n_judged': n_pref, 'excluded_pending': sum(r['judgement']['preference'] == 'pending' for r in judged),
                       'excluded_unsure': sum(r['judgement']['preference'] == 'unsure' for r in judged),
                       'ties': pref_counts['tie'], 'best_counts': {pol: pref_counts[pol] for pol in POLICIES},
                       'win_rate_best_of_three': win_rate,
                       'aligned_vs_reference': pairwise(ALIGNED, 'reference'),
                       'aligned_vs_foundation': pairwise(ALIGNED, FOUNDATION),
                       'reference_vs_foundation': pairwise('reference', FOUNDATION)},
        'hallucination_rate': hallucination,
        'format_compliance': format_ok,
        'format_human_vs_auto_disagreements': format_disagree,
        'n_notes': sum(bool(r['judgement'].get('notes')) for r in judged),
        'ties_where_two_of_three_correct': {'n': len(tie_two_of_three), 'wrong_policy_counts': dict(tie_loser_counts),
                                            'records': tie_two_of_three},
    },
    'dpo_effect': {'answers_changed_vs_reference': len(dpo_changed), 'changed_ids': dpo_changed,
                   'delta_em_aligned_minus_reference': pv['delta_em'], 'mcnemar_exact_p': pv['mcnemar_exact_p'],
                   'reference_source': SOURCE},
    'limitations': LIMITATIONS,
}

# --- Вывод (формулируется по измеренным числам, без обобщений) ---
e = objective['per_policy']
ar = blind_report['manual']['preference']['aligned_vs_reference']
significant = pv['mcnemar_exact_p'] < 0.05
conclusion = (
    f"На {n_records} изображениях id_test EM: foundation {e[FOUNDATION]['em']:.1%}, reference (SFT) "
    f"{e['reference']['em']:.1%}, aligned (SFT+DPO) {e[ALIGNED]['em']:.1%}. DPO изменил {len(dpo_changed)} из "
    f"{n_records} ответов reference; ΔEM aligned − reference = {pv['delta_em']:+.1%} "
    f"(McNemar p = {pv['mcnemar_exact_p']}) — {'значимо' if significant else 'статистически не значимо'}. "
    f"По слепым суждениям ({n_pref} записей с вынесенным предпочтением, из них tie — {pref_counts['tie']}) "
    f"aligned выбран лучшим {pref_counts[ALIGNED]} раз, reference — {pref_counts['reference']}, foundation — "
    f"{pref_counts[FOUNDATION]}; в парах aligned vs reference решено {ar['wins'] + ar['losses']} сравнений "
    f"({ar['wins']}:{ar['losses']}). Галлюцинации: foundation {hallucination[FOUNDATION]['k']}/{hallucination[FOUNDATION]['n']}, "
    f"reference {hallucination['reference']['k']}/{hallucination['reference']['n']}, aligned "
    f"{hallucination[ALIGNED]['k']}/{hallucination[ALIGNED]['n']}. Формат (только буква): foundation "
    f"{format_ok[FOUNDATION]['k']}/{format_ok[FOUNDATION]['n']}, reference {format_ok['reference']['k']}/{format_ok['reference']['n']}, "
    f"aligned {format_ok[ALIGNED]['k']}/{format_ok[ALIGNED]['n']}. "
    + (f"По EM и формату reference лучше foundation, т.е. основной сдвиг дал SFT работы 09; "
       if e['reference']['em'] >= e[FOUNDATION]['em'] and format_ok['reference']['k'] >= format_ok[FOUNDATION]['k'] else '')
    + f"DPO на label-derived парах ({DPO['max_steps']} шагов, lr {DPO['learning_rate']}) изменил политику слабо. "
    + (f"Ручной win rate смещён формой: в {len(tie_two_of_three)} записях два ответа верны, а третий нет, но "
       f"записан tie (неверная политика: {dict(tie_loser_counts)}), поэтому сравнение по выбору лучшего "
       f"занижает поражения. " if tie_two_of_three else '')
    + "Вывод касается только выбора правильной буквы в ScienceQA и не означает улучшения безопасности или "
      "общей полезности модели."
)
if FORM_STATUS != 'complete':
    conclusion = 'ПРЕДВАРИТЕЛЬНО (форма заполнена не полностью). ' + conclusion
blind_report['conclusion'] = conclusion
save_json(REPORT_PATH.name, blind_report)

# --- Печать ---
print(f"Форма: {n_complete}/{n_records} записей заполнено полностью — "
      f"{'метрики финальные' if FORM_STATUS == 'complete' else 'ручные метрики ПРЕДВАРИТЕЛЬНЫЕ'}")
print('\nEM по эталонным буквам (автоматически, не human win rate):')
table_row('', 'EM', '95% CI (Wilson)')
print('  ' + '─' * LABEL_W + '─' * COL_W + '─┼' + '─' * COL_W)
for pol in POLICIES:
    table_row(pol, f"{e[pol]['correct']}/{e[pol]['n']} = {e[pol]['em']:.1%}",
              f"CI [{e[pol]['ci95_wilson'][0]:.2f}, {e[pol]['ci95_wilson'][1]:.2f}]")

print(f"\nРучные метрики (знаменатель — вынесенные суждения; pending/unsure исключены):")
print(f"  {'политика':<12}{'лучший (из ' + str(n_pref) + ')':>16}{'галлюцинации':>16}{'формат ok':>14}")
for pol in POLICIES:
    h, f = hallucination[pol], format_ok[pol]
    print(f"  {pol:<12}{pref_counts[pol]:>11} ({pref_counts[pol] / n_pref if n_pref else 0:.1%})"
          f"{h['k']:>7}/{h['n']} ({h['rate'] or 0:.1%}){f['k']:>6}/{f['n']} ({f['rate'] or 0:.1%})")
print(f"  ничьих (tie): {pref_counts['tie']} из {n_pref}; исключено pending/unsure: "
      f"{blind_report['manual']['preference']['excluded_pending']}/{blind_report['manual']['preference']['excluded_unsure']}")
for name, key in (('aligned vs reference', 'aligned_vs_reference'), ('aligned vs foundation', 'aligned_vs_foundation'),
                  ('reference vs foundation', 'reference_vs_foundation')):
    pw = blind_report['manual']['preference'][key]
    print(f"  {name:<24} побед {pw['wins']}, поражений {pw['losses']}, не решено {pw['undecided']}"
          + (f", sign-test p = {pw['sign_test_p']}" if pw['sign_test_p'] is not None else ''))
if tie_two_of_three:
    print(f"\nВНИМАНИЕ: в {len(tie_two_of_three)} записях с tie верны два ответа из трёх — проигрыш третьего "
          f"в win rate не учтён. Неверная политика: {dict(tie_loser_counts)}")
if format_disagree:
    print(f"\nРасхождения ручной оценки формата с автоматической проверкой ({len(format_disagree)}):")
    for d in format_disagree:
        print(f"  {d['id']} {d['slot']}: ответ {d['answer']!r} — вручную {d['human']}, автоматически {d['auto']}")
print('\nВывод:\n  ' + conclusion)
print('\nОграничения label-derived предпочтений и оценки:')
for item in LIMITATIONS:
    print('  • ' + item)
print(f'\n-> {REPORT_PATH}')

Форма: 64/64 записей заполнено полностью — метрики финальные

EM по эталонным буквам (автоматически, не human win rate):
                                                              EM │95% CI (Wilson)
  ───────────────────────────────────────────────────────────────┼──────────────
  foundation                                       55/64 = 85.9% │CI [0.75, 0.92]
  reference                                        57/64 = 89.1% │CI [0.79, 0.95]
  aligned                                          58/64 = 90.6% │CI [0.81, 0.96]

Ручные метрики (знаменатель — вынесенные суждения; pending/unsure исключены):
  политика      лучший (из 64)    галлюцинации     формат ok
  foundation            4 (6.2%)      1/64 (1.6%)    59/64 (92.2%)
  reference             0 (0.0%)      0/64 (0.0%)    64/64 (100.0%)
  aligned               1 (1.6%)      0/64 (0.0%)    64/64 (100.0%)
  ничьих (tie): 59 из 64; исключено pending/unsure: 0/0
  aligned vs reference     побед 1, поражений 0, не решено 63, sign-tes

### Задание 11. Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

Гипотеза: эффект DPO сильнее на своём предмете, чем на чужом. Модель обучалась только на вопросах по естественным наукам.

**Задание 11. Творческое мини-исследование**

**Что делали**: проверили гипотезу, что эффект DPO сильнее на своём предмете. Сравнили прирост margin g на ood_test (natural science, 7–8 кл.) и general_test (social science) — дополнительных входах, не участвовавших ни в обучении, ни в основной оценке.<br>
**Зачем**: выяснить, ограничен ли метод предметом обучения (все 512 DPO-пар — natural science, 1–6 классы).<br>
**Результат**: g > 0 на всех выборках: train 2,25 (×9,5), id_test 1,81 (×6,1), ood_test 1,08 (×3,0), general_test 1,23 (×3,4). H1 не подтверждена: разность −0,15, CI [−0,48; +0,18]. H2 не подтверждена: −14,1 п.п., CI [−26,6; −1,6] — значимо в обратную сторону. Предметного ограничения не найдено, слабее всего эффект на трудных вопросах ood_test. Файлы `bonus.md`, `bonus_summary.json`.

In [12]:
# Творческое мини-исследование: переносится ли эффект DPO за пределы предмета обучения?
#
# Гипотеза сформулирована ДО измерений, только по составу train (preferences.jsonl / manifest):
# все 512 train-пар — natural science, 1–6 классы; DPO учился предпочитать верную букву именно на этих вопросах.
#   H1: прирост DPO-margin на непросмотренных парах (g = [log π_θ(y_w) − log π_θ(y_l)] − [log π_ref(y_w) − log π_ref(y_l)])
#       в том же предмете (ood_test: natural science, 7–12 классы) больше, чем в другом предмете
#       (general_test: social/language science): mean g(ood) − mean g(general) > 0.
#   H2: доля пар с g > 0 (DPO сдвинул предпочтение в сторону верной буквы) на ood_test выше, чем на general_test.
#   H0: эффект DPO не зависит от предмета (например, если DPO выучил предмет-независимый сдвиг — позицию буквы).
# Дополнительные реальные входы: ood_test и general_test (по 64 изображения из официального test ScienceQA);
# они не использовались ни в обучении, ни в основной оценке. id_test для выбора гипотезы не смотрели —
# он и train используются ниже только для сопоставления с основным экспериментом.
# Решающее правило: «подтверждено» — 95% bootstrap CI разности целиком > 0; «направление совпадает» — точечная
# оценка > 0, CI содержит 0; иначе «не подтверждено».
BONUS_DIR = ARTIFACTS
BONUS_MD = BONUS_DIR / 'bonus.md'
BONUS_LOGPS = BONUS_DIR / 'bonus_logps.jsonl'
BONUS_PREDS = BONUS_DIR / 'bonus_predictions.jsonl'
BONUS_SUMMARY = BONUS_DIR / 'bonus_summary.json'
N_BOOT = 10_000
HYPOTHESIS = {
    'H1': 'mean g(ood_test) − mean g(general_test) > 0, где g — прирост DPO-margin относительно reference',
    'H2': 'P(g > 0 | ood_test) − P(g > 0 | general_test) > 0',
    'H0': 'эффект DPO одинаков для своего и чужого предмета',
    'decision_rule': 'подтверждено — 95% bootstrap CI целиком > 0; направление совпадает — оценка > 0, CI содержит 0; '
                     'иначе не подтверждено',
    'formulated_from': 'состав train (natural science, 1–6 классы), без просмотра id_test',
}
EXTRA_SPLITS = ('ood_test', 'general_test')
COMPARE_SPLITS = ('train_seen', 'id_test')     # для сопоставления с основным экспериментом
by_split_pairs = {s: [p for p in preference_pairs if p['split'] == s] for s in EXTRA_SPLITS}
seen_ids = [train_pairs[i]['id'] for i in DATA_ORDER]
train_by_id = {p['id']: p for p in train_pairs}
by_split_pairs['train_seen'] = [train_by_id[i] for i in dict.fromkeys(seen_ids)][:64]   # пары, которые DPO видел
by_split_pairs['id_test'] = list(test_pairs)
ALL_BONUS_SPLITS = (*EXTRA_SPLITS, *COMPARE_SPLITS)

# Дополнительные входы не пересекаются с train и с id_test
extra_hashes = {p['image_sha256'] for s in EXTRA_SPLITS for p in by_split_pairs[s]}
assert not extra_hashes & {p['image_sha256'] for p in train_pairs}, 'дополнительные входы пересекаются с train'
assert not extra_hashes & {p['image_sha256'] for p in test_pairs}, 'дополнительные входы пересекаются с id_test'

aligned_sha = params_sha256(adapter_params(policy_model, POLICY))
assert aligned_sha == expected_sha, 'активен не обученный адаптер'

def read_jsonl(path):
    return [json.loads(s) for s in path.read_text(encoding='utf-8').splitlines() if s.strip()] if path.is_file() else []

# --- 1. log p chosen/rejected под policy и reference (кэш привязан к sha256 адаптеров) ---
@torch.no_grad()
def pair_logps(pair):
    image = load_image(pair['image'])
    inp = {k: prepare_answer(pair, pair[k], image) for k in ('chosen', 'rejected')}
    out = {}
    for name, adapter in (('policy', POLICY), ('ref', REF)):
        policy_model.set_adapter(adapter)
        for k in ('chosen', 'rejected'):
            out[f'{name}_{k}'] = completion_logprob(policy_model, inp[k]).item()
    policy_model.set_adapter(POLICY); freeze_reference(policy_model)
    return out

policy_model.eval()
cached = {(r['split'], r['id']): r for r in read_jsonl(BONUS_LOGPS)
          if r.get('aligned_sha256') == aligned_sha and r.get('ref_sha256') == REF_SHA256}
logp_rows = []
t0 = time.time()
for s in ALL_BONUS_SPLITS:
    for p in by_split_pairs[s]:
        r = cached.get((s, p['id']))
        if r is None:
            r = {'split': s, 'id': p['id'], 'subject': p['subject'], 'grade': p['grade'], 'n_choices': len(p['choices']),
                 'chosen_index': p['chosen_index'], 'rejected_index': p['rejected_index'],
                 'aligned_sha256': aligned_sha, 'ref_sha256': REF_SHA256, **pair_logps(p)}
        r['policy_margin'] = r['policy_chosen'] - r['policy_rejected']
        r['ref_margin'] = r['ref_chosen'] - r['ref_rejected']
        r['g'] = r['policy_margin'] - r['ref_margin']
        logp_rows.append(r)
    print(f'  log p: {s} готов ({time.time() - t0:.0f} с)')
BONUS_LOGPS.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in logp_rows), encoding='utf-8')

# --- 2. Ответы трёх политик на дополнительных входах (EM, число изменённых DPO ответов) ---
cached_pred = {r['id']: r for r in read_jsonl(BONUS_PREDS) if r.get('aligned_sha256') == aligned_sha}
pred_rows = []
for s in EXTRA_SPLITS:
    for p in by_split_pairs[s]:
        r = cached_pred.get(p['id'])
        if r is None:
            image = load_image(p['image'])
            ans = {FOUNDATION: generate_answer(p, FOUNDATION, image), 'reference': generate_answer(p, REF, image),
                   ALIGNED: generate_answer(p, POLICY, image)}
            r = {'split': s, 'id': p['id'], 'gold': p['chosen'], 'n_choices': len(p['choices']), 'answers': ans,
                 'letters': {k: parse_letter(v, len(p['choices'])) for k, v in ans.items()},
                 'aligned_sha256': aligned_sha}
        pred_rows.append(r)
    print(f'  ответы: {s} готов ({time.time() - t0:.0f} с)')
BONUS_PREDS.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in pred_rows), encoding='utf-8')

# --- 3. Статистика ---
rng = np.random.default_rng(SEED)
g_by = {s: np.array([r['g'] for r in logp_rows if r['split'] == s]) for s in ALL_BONUS_SPLITS}

def boot_ci(stat):
    vals = np.array([stat() for _ in range(N_BOOT)])
    return [round(float(np.percentile(vals, 2.5)), 4), round(float(np.percentile(vals, 97.5)), 4)]

def diff_test(f, a, b):
    """Разность статистики f между независимыми выборками a и b с bootstrap-CI."""
    est = f(a) - f(b)
    ci = boot_ci(lambda: f(rng.choice(a, len(a))) - f(rng.choice(b, len(b))))
    verdict = ('подтверждено' if ci[0] > 0 else 'направление совпадает' if est > 0 else 'не подтверждено')
    return {'estimate': round(float(est), 4), 'ci95': ci, 'verdict': verdict}

mean = lambda x: float(np.mean(x))
pos_share = lambda x: float(np.mean(x > 0))
per_split = {}
for s in ALL_BONUS_SPLITS:
    g = g_by[s]
    per_split[s] = {'n': int(len(g)), 'mean_g': round(mean(g), 4), 'ci95_mean_g': boot_ci(lambda: mean(rng.choice(g, len(g)))),
                    'share_g_pos': round(pos_share(g), 4), 'median_g': round(float(np.median(g)), 4),
                    'mean_ref_margin': round(mean([r['ref_margin'] for r in logp_rows if r['split'] == s]), 4)}
for s in EXTRA_SPLITS:
    rows = [r for r in pred_rows if r['split'] == s]
    per_split[s]['em'] = {pol: round(np.mean([r['letters'][pol] == r['gold'] for r in rows]), 4) for pol in POLICIES}
    per_split[s]['dpo_changed_answers'] = sum(r['answers']['reference'] != r['answers'][ALIGNED] for r in rows)
    per_split[s]['subjects'] = dict(Counter(p['subject'] for p in by_split_pairs[s]))
per_split['id_test']['em'] = {pol: objective['per_policy'][pol]['em'] for pol in POLICIES}
per_split['id_test']['dpo_changed_answers'] = len(dpo_changed)

tests = {'H1': diff_test(mean, g_by['ood_test'], g_by['general_test']),
         'H2': diff_test(pos_share, g_by['ood_test'], g_by['general_test'])}

# Альтернативное объяснение (H0): предмет-независимый сдвиг по позиции буквы.
# rejected = следующий вариант по кругу, поэтому верная буква обычно стоит РАНЬШЕ rejected.
def position_split(s):
    rows = [r for r in logp_rows if r['split'] == s]
    before = [r['g'] for r in rows if r['chosen_index'] < r['rejected_index']]
    after = [r['g'] for r in rows if r['chosen_index'] > r['rejected_index']]
    return {'chosen_before_rejected': {'n': len(before), 'mean_g': round(mean(before), 4) if before else None},
            'chosen_after_rejected': {'n': len(after), 'mean_g': round(mean(after), 4) if after else None}}
position = {s: position_split(s) for s in ALL_BONUS_SPLITS}
train_before_share = round(np.mean([p['chosen_index'] < p['rejected_index'] for p in train_pairs]), 4)

bonus_summary = {'hypothesis': HYPOTHESIS, 'n_boot': N_BOOT, 'seed': SEED, 'aligned_sha256': aligned_sha,
                 'reference_sha256': REF_SHA256, 'per_split': per_split, 'tests': tests,
                 'position_diagnostic': {'train_share_chosen_before_rejected': train_before_share, 'by_split': position},
                 'files': {'logps': str(BONUS_LOGPS), 'predictions': str(BONUS_PREDS)}}
save_json(BONUS_SUMMARY.name, bonus_summary)

# --- 4. bonus.md ---
P = per_split
def fmt_ci(ci): return f'[{ci[0]:+.3f}; {ci[1]:+.3f}]'
def pp(x): return f'{x * 100:+.1f} п.п.'                      # разность долей -> процентные пункты
def odds_pct(x): return f'{math.exp(x) - 1:+.0%}'               # разность log-margin -> изменение отношения шансов
def times(x): return f'×{math.exp(x):.1f}'                      # g -> во сколько раз выросло отношение верный/неверный
split_names = {'train_seen': 'train (пары, виденные DPO)', 'id_test': 'id_test (основной эксперимент)',
               'ood_test': 'ood_test — свой предмет, 7–8 кл.', 'general_test': 'general_test — чужой предмет'}
table = '\n'.join(
    f"| {split_names[s]} | {P[s]['n']} | {P[s]['mean_g']:+.3f} {fmt_ci(P[s]['ci95_mean_g'])} | {P[s]['share_g_pos']:.0%} | "
    f"{P[s]['mean_ref_margin']:.2f} | " + (f"{P[s]['em'][FOUNDATION]:.1%} / {P[s]['em']['reference']:.1%} / {P[s]['em'][ALIGNED]:.1%} | "
                                            f"{P[s]['dpo_changed_answers']}" if 'em' in P[s] else '— | —') + ' |'
    for s in ('train_seen', 'id_test', 'ood_test', 'general_test'))
pos_table = '\n'.join(
    f"| {split_names[s]} | {position[s]['chosen_before_rejected']['n']} / {position[s]['chosen_before_rejected']['mean_g']:+.3f} | "
    f"{position[s]['chosen_after_rejected']['n']} / "
    + (f"{position[s]['chosen_after_rejected']['mean_g']:+.3f}" if position[s]['chosen_after_rejected']['mean_g'] is not None else '—') + ' |'
    for s in ALL_BONUS_SPLITS)

h1, h2 = tests['H1'], tests['H2']
g_ood, g_gen = P['ood_test']['mean_g'], P['general_test']['mean_g']
explain = []
if P['train_seen']['mean_g'] > max(g_ood, g_gen, P['id_test']['mean_g']):
    explain.append('Сильнее всего margin вырос на парах, которые DPO видел при обучении; на непросмотренных — '
                   'слабее: за 40 шагов (0.6 эпохи, lr 5e-6) DPO лишь частично обобщился.')
if g_gen > 0 and h1['verdict'] != 'подтверждено':
    explain.append('Прирост на чужом предмете тоже положителен: часть эффекта DPO не привязана к содержанию '
                   'вопроса — это согласуется с H0 (общий сдвиг к выбору верной буквы / формату ответа).')
if g_gen <= 0:
    explain.append('На чужом предмете прирост не наблюдается: эффект DPO ограничен предметом обучения.')
cb = [position[s]['chosen_before_rejected']['mean_g'] for s in EXTRA_SPLITS]
ca = [position[s]['chosen_after_rejected']['mean_g'] for s in EXTRA_SPLITS]
if all(x is not None for x in cb + ca) and all(b > a for b, a in zip(cb, ca)):
    explain.append(f'В обоих дополнительных сплитах прирост больше, когда верная буква стоит раньше rejected; '
                   f'в train так в {train_before_share:.0%} пар — возможен позиционный shortcut из-за правила '
                   '«rejected = следующий вариант».')
explain.append(f"EM почти не меняется: DPO изменил {P['ood_test']['dpo_changed_answers']} из {P['ood_test']['n']} ответов "
               f"на ood_test и {P['general_test']['dpo_changed_answers']} из {P['general_test']['n']} на general_test "
               f"(в основном эксперименте — {P['id_test']['dpo_changed_answers']} из {P['id_test']['n']}): сдвиг margin "
               'мал по сравнению с исходным отрывом верной буквы, поэтому greedy-ответы в основном те же.')

md = f"""# Бонус: переносится ли эффект DPO за пределы предмета обучения

## Гипотеза
Сформулирована до измерений, только по составу train: все 512 DPO-пар — natural science, 1–6 классы.

- **H1:** прирост DPO-margin на непросмотренных парах больше в своём предмете (ood_test: natural science, 7–12 кл.),
  чем в чужом (general_test: social/language science): mean g(ood) − mean g(general) > 0.
- **H2:** доля пар с g > 0 на ood_test выше, чем на general_test.
- **H0:** эффект DPO не зависит от предмета.

Здесь g = [log π_θ(y_w|x) − log π_θ(y_l|x)] − [log π_ref(y_w|x) − log π_ref(y_l|x)] — насколько DPO усилил
предпочтение верной буквы относительно reference (SFT работы 09). Пары строятся так же, как для обучения:
chosen — верная буква, rejected — следующий неверный вариант.

## Протокол
- Дополнительные реальные входы: **ood_test** и **general_test** (по 64 изображения из официального test ScienceQA,
  ревизия `{SCIENCEQA_REVISION[:12]}`); пересечений с train и id_test по sha256 изображений нет (проверено).
- id_test в выборе гипотезы не участвовал; он и 64 train-пары, виденные DPO, используются только для сопоставления.
- log p — `completion_logprob` (токены ответа и окончания), адаптеры `{POLICY}` (aligned, sha256 `{aligned_sha[:12]}…`)
  и `{REF}` (reference, sha256 `{REF_SHA256[:12]}…`); ответы — greedy, max_new_tokens = {GEN['max_new_tokens']}.
- Статистика: bootstrap ({N_BOOT} повторов, seed {SEED}), 95% перцентильный CI. Решающее правило: {HYPOTHESIS['decision_rule']}.
- Воспроизведение: выполнить ноутбук до этой ячейки; сырые данные — `{BONUS_LOGPS.name}`, `{BONUS_PREDS.name}`,
  сводка — `{BONUS_SUMMARY.name}`.

## Наблюдения
| Выборка | n | mean g [95% CI] | доля g > 0 | ref margin | EM foundation / reference / aligned | ответов изменено DPO |
|---|---|---|---|---|---|---|
{table}

| Проверка | Оценка | 95% CI | Вердикт |
|---|---|---|---|
| H1: mean g(ood) − mean g(general) | {h1['estimate']:+.4f} (отношение шансов {odds_pct(h1['estimate'])}) | {fmt_ci(h1['ci95'])} ({odds_pct(h1['ci95'][0])} … {odds_pct(h1['ci95'][1])}) | **{h1['verdict']}** |
| H2: P(g>0 \\| ood) − P(g>0 \\| general) | {h2['estimate']:+.4f} ({pp(h2['estimate'])}) | {fmt_ci(h2['ci95'])} ({pp(h2['ci95'][0])} … {pp(h2['ci95'][1])}) | **{h2['verdict']}** |

Диагностика позиции (n / mean g): rejected — следующий вариант по кругу, поэтому верная буква обычно стоит раньше
(в train — {train_before_share:.0%} пар).

| Выборка | chosen раньше rejected | chosen позже rejected |
|---|---|---|
{pos_table}

## Сопоставление с основным экспериментом и расхождения
""" + '\n'.join(f'- {x}' for x in explain) + f"""

## Вывод
H1 — **{h1['verdict']}**, H2 — **{h2['verdict']}**. Вывод относится к сдвигу log-вероятностей ответа-буквы на
вопросах ScienceQA при label-derived предпочтениях (верная буква против следующего варианта); он не говорит о
безопасности или общей полезности модели. Ограничения: по 64 пары на выборку, один запуск DPO (один seed),
предметы general_test почти целиком social science с 4 вариантами, а ood_test отличается и классом, и числом вариантов,
поэтому эффект предмета частично смешан с форматом вопроса.
"""
BONUS_MD.write_text(md, encoding='utf-8')

print(f'\nH1 (ood − general, mean g): {h1["estimate"]:+.4f} (отношение шансов {odds_pct(h1["estimate"])}) '
      f'CI {fmt_ci(h1["ci95"])} ({odds_pct(h1["ci95"][0])} … {odds_pct(h1["ci95"][1])}) -> {h1["verdict"]}')
print(f'H2 (ood − general, доля g>0): {h2["estimate"]:+.4f} ({pp(h2["estimate"])}) '
      f'CI {fmt_ci(h2["ci95"])} ({pp(h2["ci95"][0])} … {pp(h2["ci95"][1])}) -> {h2["verdict"]}')
print('  п.п. — процентные пункты (разность двух долей); отношение шансов — во сколько изменилось '
      'отношение p(верный)/p(неверный): −14% значит на ood прирост в 0.86 раза от прироста на general')
for s in ('train_seen', 'id_test', 'ood_test', 'general_test'):
    print(f"  {split_names[s]:<36} mean g = {P[s]['mean_g']:+.3f} ({times(P[s]['mean_g'])}) {fmt_ci(P[s]['ci95_mean_g'])}, доля g>0 = {P[s]['share_g_pos']:.0%}"
          + (f", EM ref/aligned = {P[s]['em']['reference']:.1%}/{P[s]['em'][ALIGNED]:.1%}, изменено {P[s]['dpo_changed_answers']}"
             if 'em' in P[s] else ''))
print(f'-> {BONUS_MD}')

  log p: ood_test готов (0 с)
  log p: general_test готов (0 с)
  log p: train_seen готов (0 с)
  log p: id_test готов (0 с)
  ответы: ood_test готов (0 с)
  ответы: general_test готов (0 с)

H1 (ood − general, mean g): -0.1465 (отношение шансов -14%) CI [-0.477; +0.182] (-38% … +20%) -> не подтверждено
H2 (ood − general, доля g>0): -0.1406 (-14.1 п.п.) CI [-0.266; -0.016] (-26.6 п.п. … -1.6 п.п.) -> не подтверждено
  п.п. — процентные пункты (разность двух долей); отношение шансов — во сколько изменилось отношение p(верный)/p(неверный): −14% значит на ood прирост в 0.86 раза от прироста на general
  train (пары, виденные DPO)           mean g = +2.250 (×9.5) [+2.008; +2.481], доля g>0 = 95%
  id_test (основной эксперимент)       mean g = +1.812 (×6.1) [+1.549; +2.068], доля g>0 = 89%, EM ref/aligned = 89.1%/90.6%, изменено 1
  ood_test — свой предмет, 7–8 кл.     mean g = +1.082 (×3.0) [+0.789; +1.371], доля g>0 = 78%, EM ref/aligned = 70.3%/70.3%, изменено 3
  general_test — чужой пр

**Что такое g.** Для каждой пары «верная буква $y_w$ / неверная буква $y_l$» g показывает, насколько DPO изменил предпочтение модели в пользу верного ответа по сравнению с reference (SFT работы 09):

$$g = \underbrace{\big[\log \pi_\theta(y_w|x) - \log \pi_\theta(y_l|x)\big]}_{\text{margin после DPO (aligned)}} - \underbrace{\big[\log \pi_{ref}(y_w|x) - \log \pi_{ref}(y_l|x)\big]}_{\text{margin до DPO (reference)}}$$

$e^{g}$ — во сколько раз после DPO изменилось отношение $p(\text{верный}) / p(\text{неверный})$ (в выводе — «×N»). Например, $g = 1.1$ → $\times 3.0$.

**Как читать знак:**
- $g > 0$ — DPO усилил предпочтение верного ответа;
- $g = 0$ — DPO ничего не изменил;
- $g < 0$ — DPO сдвинул модель в сторону неверного ответа.

g чувствительнее EM: EM меняется, только если модель сменила выбранную букву, а g фиксирует и рост уверенности в уже верном ответе.

## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.

In [13]:
# Анализ ошибок по сохранённым предсказаниям: blind_predictions.jsonl (id_test, три политики),
# bonus_predictions.jsonl (ood_test, general_test), manifest (вопросы, классы, число вариантов).
# Модель не запускается — только чтение артефактов полного режима.
from collections import defaultdict
assert not FAST_DEV_RUN, 'итоговый анализ — только по полному режиму (FAST_DEV_RUN=False)'
ERROR_PATH = ARTIFACTS / 'error_analysis.json'
rec = {r['id']: r for r in records}

def read_jsonl(path):
    return [json.loads(s) for s in path.read_text(encoding='utf-8').splitlines() if s.strip()]

rows_by_split = {'id_test': [{'split': 'id_test', **r} for r in read_jsonl(BLIND_PATH)]}
bonus_file = ARTIFACTS / 'bonus_predictions.jsonl'
if bonus_file.is_file():
    for r in read_jsonl(bonus_file):
        rows_by_split.setdefault(r['split'], []).append(r)
SPLITS_A = [s for s in ('id_test', 'ood_test', 'general_test') if s in rows_by_split]

# Тип задачи: семейства natural science из аудита пар + шаблоны social/language science
EXTRA_FAMILIES = [('genetics_punnett', r'expected ratio of offspring|probability that .* (cross|offspring)|homozygous|heterozygous'),
                  ('experiment_design', r'experiment can best answer|test show|independent variable|dependent variable'),
                  ('map_geography', r'highlighted|colony|which (ocean|continent|state)|capital of'),
                  ('economics_trade', r'trade|supply|demand|opportunity cost'),
                  ('language', r'which word|figure of speech|tone|dictionary')]
def task_of(qid):
    q = rec[qid]['question'].split('\n')[0].lower()
    return next((n for n, pat in FAMILIES + EXTRA_FAMILIES if re.search(pat, q)), 'other')

def correct(r, pol): return r['letters'][pol] == r['gold']

analysis = {'policies': list(POLICIES), 'splits': {}}
for s in SPLITS_A:
    xs = rows_by_split[s]
    e = {'n': len(xs), 'em': {}, 'lenient_vs_strict': {}, 'errors_by_task': {}, 'em_by_n_choices': {}, 'em_by_grade': {}}
    for pol in POLICIES:
        e['em'][pol] = round(np.mean([correct(r, pol) for r in xs]), 4)
        strict = np.mean([r['answers'][pol].strip() == r['gold'] for r in xs])
        e['lenient_vs_strict'][pol] = {'lenient_em': e['em'][pol], 'strict_em': round(float(strict), 4),
                                       'non_letter_answers': dict(Counter(r['answers'][pol] for r in xs
                                                                          if not strict_format(r['answers'][pol], len(rec[r['id']]['choices']))).most_common(5))}
        e['errors_by_task'][pol] = dict(Counter(task_of(r['id']) for r in xs if not correct(r, pol)).most_common())
        for key, fn in (('em_by_n_choices', lambda r: len(rec[r['id']]['choices'])), ('em_by_grade', lambda r: rec[r['id']]['grade'])):
            groups = defaultdict(list)
            for r in xs: groups[str(fn(r))].append(correct(r, pol))
            e[key][pol] = {k: {'n': len(v), 'correct': int(sum(v))} for k, v in sorted(groups.items())}
    # Переходы: SFT (foundation -> reference) и DPO (reference -> aligned)
    def transitions(a, b):
        fixed = [r['id'] for r in xs if not correct(r, a) and correct(r, b)]
        broken = [r['id'] for r in xs if correct(r, a) and not correct(r, b)]
        changed = [r['id'] for r in xs if r['answers'][a] != r['answers'][b]]
        return {'answers_changed': len(changed), 'fixed': fixed, 'broken': broken}
    e['sft_effect'] = transitions(FOUNDATION, 'reference')
    e['dpo_effect'] = transitions('reference', ALIGNED)
    e['all_wrong'] = [{'id': r['id'], 'task': task_of(r['id']), 'gold': r['gold'],
                       'answers': {p: r['answers'][p] for p in POLICIES},
                       'question': rec[r['id']]['question'].split('\n')[0][:90]}
                      for r in xs if not any(correct(r, p) for p in POLICIES)]
    analysis['splits'][s] = e
save_json(ERROR_PATH.name, analysis)

# --- Печать ---
for s in SPLITS_A:
    e = analysis['splits'][s]
    print(f"\n=== {s} (n = {e['n']}) ===")
    print('  EM (первая допустимая буква) / строгий EM (ответ == буква):  ' +
          ' | '.join(f"{p} {e['em'][p]:.1%}/{e['lenient_vs_strict'][p]['strict_em']:.1%}" for p in POLICIES))
    print(f"  SFT (foundation→reference): исправлено {len(e['sft_effect']['fixed'])}, сломано {len(e['sft_effect']['broken'])}, "
          f"изменено ответов {e['sft_effect']['answers_changed']}")
    print(f"  DPO (reference→aligned):    исправлено {len(e['dpo_effect']['fixed'])}, сломано {len(e['dpo_effect']['broken'])}, "
          f"изменено ответов {e['dpo_effect']['answers_changed']}"
          + (f"  {e['dpo_effect']['fixed'] + e['dpo_effect']['broken']}" if e['dpo_effect']['fixed'] + e['dpo_effect']['broken'] else ''))
    print(f"  Ошибки aligned по типам задач: {e['errors_by_task'][ALIGNED]}")
    print('  EM aligned по числу вариантов: ' + ', '.join(f"{k} вар.: {v['correct']}/{v['n']}" for k, v in e['em_by_n_choices'][ALIGNED].items()))
    print(f"  Все три политики ошиблись: {len(e['all_wrong'])}")
    for w in e['all_wrong'][:6]:
        print(f"    {w['id']:<22} {w['task']:<26} верно {w['gold']}, ответы {list(w['answers'].values())} — {w['question']}")
print(f'\n-> {ERROR_PATH}')


=== id_test (n = 64) ===
  EM (первая допустимая буква) / строгий EM (ответ == буква):  foundation 85.9%/81.2% | reference 89.1%/89.1% | aligned 90.6%/90.6%
  SFT (foundation→reference): исправлено 6, сломано 4, изменено ответов 14
  DPO (reference→aligned):    исправлено 1, сломано 0, изменено ответов 1  ['scienceqa-test-349']
  Ошибки aligned по типам задач: {'weather_vs_climate': 2, 'solution_concentration': 1, 'magnet_force': 1, 'other': 1, 'magnet_attract_repel': 1}
  EM aligned по числу вариантов: 2 вар.: 32/36, 3 вар.: 25/27, 4 вар.: 1/1
  Все три политики ошиблись: 2
    scienceqa-test-49      other                      верно A, ответы ['B.', 'B', 'B'] — Is Rafflesia arnoldii made up of many cells?
    scienceqa-test-80      magnet_attract_repel       верно B, ответы ['A', 'A', 'A'] — Will these magnets attract or repel each other?

=== ood_test (n = 64) ===
  EM (первая допустимая буква) / строгий EM (ответ == буква):  foundation 57.8%/54.7% | reference 70.3%/70.3% | aligned 

### Анализ результатов (полный режим, `FAST_DEV_RUN=False`)

Все числа взяты из сохранённых `dpo_metrics.json`, `objective_scores.json`, `blind_report.json`, `error_analysis.json` и `bonus_summary.json`. Отложенная выборка — `id_test` (64 изображения natural science, 1–6 классы). Она не использовалась ни при обучении, ни при выборе гипотезы бонуса.

**Обучение DPO.** 40 optimizer steps × 8 пар = 320 пар (0,6 эпохи train), β = 0,1, AdamW lr = 5e-6. Обучение заняло 18,4 мин на RTX 3060, пик памяти — 3,4 ГБ.
- Loss: 0,693 (log 2, политика = reference) → 0,608. Среднее первых и последних 5 шагов — 0,691 → 0,616. Все значения конечны.
- Разница margin политики и reference на обучающих парах выросла с 0 до ≈ 1,8.
- Reference не изменился (sha256 совпадает), веса LoRA сдвинулись максимум на 2·10⁻⁴.

**Качество на `id_test`** (EM по эталонной букве — автоматическая метрика, не human win rate; в скобках — 95 % CI Уилсона)

| политика | EM | строгий EM (ответ == буква) | только буква (формат) |
|---|---:|---:|---:|
| foundation (Qwen2.5-VL-3B-Instruct) | 55/64 = 85,9 % [0,75–0,92] | 81,2 % | 89,1 % |
| reference (SFT работы 09) | 57/64 = 89,1 % [0,79–0,95] | 89,1 % | 100 % |
| aligned (SFT + DPO) | 58/64 = 90,6 % [0,81–0,96] | 90,6 % | 100 % |

- **SFT** (foundation → reference) изменил 14 ответов: исправил 6, сломал 4 и убрал нарушения формата: у foundation 6 ответов с точкой и 1 несуществующая буква.
- **DPO** (reference → aligned) изменил **1 ответ из 64** (`scienceqa-test-349`, B → C, исправление). ΔEM = +1,6 п.п., McNemar p = 1,0 — **статистически не значимо**.
- **Ручная слепая оценка** (64/64 записи, один оценщик):
  - ничья (tie) в 59 записях; лучшим выбран foundation 4 раза, reference 0, aligned 1;
  - галлюцинация — только у foundation (1/64, «C» при двух вариантах);
  - формат «ok»: 59/64, 64/64, 64/64.

  Ручной win rate смещён формой «один лучший или tie»: в 6 записях верны два ответа из трёх (во всех ошибался foundation), но они записаны как tie. Поэтому поражения foundation в win rate не попали.

**Разбор ошибок** (`error_analysis.json`)
- **id_test:** у aligned 6 ошибок — «погода или климат» (2), концентрация частиц, сила магнитов, притяжение магнитов, прочее. Все три политики ошиблись дважды: `scienceqa-test-49` (многоклеточна ли раффлезия — все ответили B) и `scienceqa-test-80` (магниты — все ответили A).
- **ood_test** (natural science; в выборке фактически 7–8 классы): EM 57,8 % → 70,3 % → 70,3 %.
  - SFT исправил 11 и сломал 3 ответа, DPO исправил 1 и сломал 1.
  - 19 ошибок aligned сосредоточены в задачах старших классов: дизайн эксперимента (7), генетика и решётка Пеннета (5), сила магнитов (4).
  - На вопросах с 5 вариантами верно только 2 из 7: таких вопросов в train нет (там 2–4 варианта).
  - Все три политики ошиблись в 15 вопросах.
- **general_test** (social science): у всех трёх политик одинаковые ответы и EM 85,9 %. Ни SFT, ни DPO не изменили ни одного ответа. Ошибки — карты («какая страна/океан выделены», 7) и экономика (обмен, 2).
- **Бонус:** прирост DPO-margin g > 0 на всех выборках:

  | выборка | g | рост отношения «верный / неверный» |
  |---|---:|---:|
  | train | 2,25 | ×9,5 |
  | id_test | 1,81 | ×6,1 |
  | ood_test | 1,08 | ×3,0 |
  | general_test | 1,23 | ×3,4 |

  Гипотеза о предметном ограничении не подтвердилась: доля пар с g > 0 на чужом предмете даже выше (92 % против 78 %). Слабее всего DPO действует на трудных для модели вопросах ood_test.

**Ограничения выборки и оценки**
- По 64 вопроса на выборку: ширина 95 % CI около ±8–10 п.п., поэтому различия в 1–2 ответа неразличимы. Один запуск DPO (один seed).
- ood_test отличается от train и классом, и числом вариантов (3–5). general_test почти целиком состоит из вопросов social science с 4 вариантами. Эффекты предмета, трудности и формата смешаны.
- Предпочтения label-derived (верная буква против следующего варианта), а не экспертные. Ручную оценку выполнил один оценщик, которому была видна эталонная буква.
- EM считается по первой допустимой букве ответа. Строгий EM и ручной формат приведены отдельно.

**Вывод.** Основной сдвиг качества и формата дал SFT работы 09: +3,2 п.п. EM на id_test, +12,5 п.п. на ood_test и 100 % ответов-букв. DPO на label-derived парах при этом бюджете действительно усилил предпочтение верной буквы в log-вероятностях: g > 0 у 89 % пар id_test. Но greedy-ответы он почти не изменил (1 из 64), и прирост EM статистически не значим. Вывод касается только выбора правильной буквы в ScienceQA. Он не означает улучшения безопасности, развёрнутых ответов или общей полезности модели.

In [14]:
# Пакет для сдачи: проверка полного режима и реальных измерений, версии зависимостей, перечень артефактов, инструкции
import subprocess, sys, platform, importlib.metadata as md

# 1) Полный режим и реальные измерения
assert not FAST_DEV_RUN, 'для сдачи нужен полный режим (FAST_DEV_RUN=False)'
load = lambda name: json.loads((ARTIFACTS / name).read_text(encoding='utf-8'))
dm = load('dpo_metrics.json')
assert dm['status'] == 'completed' and dm['is_full_experiment'] and not dm['fast_dev_run'], 'DPO не в полном режиме'
assert dm['budget']['done_optimizer_steps'] == DPO['max_steps'] == 40 and len(dm['history']) == 40
assert dm['all_losses_finite'] and dm['reference_unchanged']
assert (ARTIFACTS / 'aligned_adapter' / 'train' / 'adapter_model.safetensors').is_file(), 'нет обученного адаптера'
blind_rows = read_jsonl(BLIND_PATH)
assert len(blind_rows) == 64 and all(set(r['answers']) == set(POLICIES) for r in blind_rows), 'не у всех ID три ответа'
judged_rows = read_jsonl(JUDGE_PATH)
assert len(judged_rows) == 64 and all(is_complete(r) for r in judged_rows), 'форма ручной проверки заполнена не полностью'
REQUIRED = ['preference_pairs_check.json', 'preferences.jsonl', 'pair_audit.json', 'reference_policy.json',
            'logprob_check.json', 'dpo_protocol.json', 'dpo_metrics.json', 'dpo_steps_train.jsonl',
            'baseline_predictions.jsonl', 'baseline_info.json', 'blind_predictions.jsonl', 'objective_scores.json',
            'blind_judgements.jsonl', 'blind_report.json', 'error_analysis.json']
missing = [f for f in REQUIRED if not (ARTIFACTS / f).is_file()]
assert not missing, f'нет артефактов: {missing}'
BONUS_FILES = [f for f in ('bonus.md', 'bonus_summary.json', 'bonus_logps.jsonl', 'bonus_predictions.jsonl')
               if (ARTIFACTS / f).is_file()]

# 2) Версии зависимостей (окружение ядра: в нём шли DPO-обучение и инференс)
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True).stdout
(ARTIFACTS / 'requirements-notebook.txt').write_text(freeze, encoding='utf-8')
def v(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return None
props = torch.cuda.get_device_properties(0)
versions = {
    'python': platform.python_version(), 'platform': platform.platform(),
    'gpu': {'name': props.name, 'total_memory_gb': round(props.total_memory / 2**30, 2), 'cuda': torch.version.cuda,
            'bf16_supported': torch.cuda.is_bf16_supported()},
    'packages': {p: v(p) for p in ('torch', 'transformers', 'peft', 'trl', 'accelerate', 'bitsandbytes', 'datasets',
                                   'safetensors', 'ipywidgets', 'scipy', 'numpy', 'pillow')},
    'model': MODEL_ID, 'quantization': '4-bit nf4 + double quant (bitsandbytes)',
    'dataset': {'id': SCIENCEQA_ID, 'revision': SCIENCEQA_REVISION, 'data_dir': str(DATA_DIR)},
    'reference': {'source': SOURCE, 'adapter': str(LAB9_ADAPTER), 'params_sha256': REF_SHA256},
    'aligned': {'adapter': str(ARTIFACTS / 'aligned_adapter' / 'train'),
                'params_sha256': dm['adapter']['policy_params_sha256'], 'file_sha256': dm['adapter']['file_sha256']},
    'fast_dev_run': FAST_DEV_RUN, 'seed': SEED,
}
save_json('version.json', versions)

# 3) Перечень артефактов с размерами и SHA-256
def sha256_file(path, limit=300 * 2**20):
    if path.stat().st_size > limit:
        return None
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(2**20), b''):
            h.update(chunk)
    return h.hexdigest()
files = [{'path': p.relative_to(ARTIFACTS).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_file(p)}
         for p in sorted(ARTIFACTS.rglob('*')) if p.is_file() and p.name != 'submission_manifest.json']
save_json('submission_manifest.json', {'artifacts_dir': str(ARTIFACTS), 'n_files': len(files), 'files': files})

# 4) Инструкции повторного запуска
obj, rep = load('objective_scores.json'), load('blind_report.json')
em_ = obj['per_policy']
REPRO = f"""# Лабораторная 11 — повторный запуск

## Окружение
- Python {versions['python']}, {versions['platform']}; GPU: {props.name} ({versions['gpu']['total_memory_gb']} ГБ), CUDA {torch.version.cuda}.
- Ядро ноутбука — `.venv`; полный список пакетов — `requirements-notebook.txt`, ключевые версии — `version.json`
  (torch {versions['packages']['torch']}, transformers {versions['packages']['transformers']}, peft {versions['packages']['peft']},
  trl {versions['packages']['trl']}, bitsandbytes {versions['packages']['bitsandbytes']}).
- Установка: `.venv\\Scripts\\python.exe -m pip install -r artifacts/lab11/requirements-notebook.txt`.

## Данные и исходная политика
- ScienceQA `{SCIENCEQA_ID}@{SCIENCEQA_REVISION}` через `prepare_scienceqa` в `{DATA_DIR}` (те же изображения, что в работе 09).
- Reference и стартовая политика — SFT-адаптер работы 09 `{LAB9_ADAPTER}` (base `{MODEL_ID}`, 4-bit nf4).
  Без него обе политики стартуют с опубликованной instruct-модели (это не продолжение собственного SFT).

## Порядок запуска
1. Первая ячейка: `FAST_DEV_RUN = False` (smoke-режим `True` — 2 шага DPO, результаты в `aligned_adapter/smoke`, не для сдачи).
2. Выполнить ячейки сверху вниз. Долгие шаги кэшируются:
   - DPO-обучение (~{dm['train_time_s'] / 60:.0f} мин) не повторяется, если в `aligned_adapter/train` есть завершённый запуск
     (`FORCE_RETRAIN = True` — переобучить);
   - ответы на тесте берутся из `baseline_predictions.jsonl` / `blind_predictions.jsonl`, если совпадают ID, изображения
     и sha256 адаптеров (`FORCE_REGENERATE`, `FORCE_REGENERATE_ALIGNED` — пересчитать);
   - бонус кэширует `bonus_logps.jsonl` / `bonus_predictions.jsonl` по sha256 адаптеров.
3. Форма ручной проверки (`blind_judgements.jsonl`) сохраняет заполненные суждения; перед сдачей
   `REQUIRE_COMPLETE_FORM = True`.

## Ключевые настройки
- DPO: β = {DPO['beta']}, AdamW lr = {DPO['learning_rate']}, accumulation = {DPO['gradient_accumulation_steps']},
  {DPO['max_steps']} optimizer steps ({DPO['pairs_seen']} пар), seed {SEED}, gradient checkpointing, dropout = 0.
- Декодирование: greedy, max_new_tokens = {GEN['max_new_tokens']}, prompt отрезается.

## Итоговые метрики (id_test, n = 64)
- EM: foundation {em_['foundation']['em']:.1%}, reference {em_['reference']['em']:.1%}, aligned {em_['aligned']['em']:.1%}
  (ΔEM aligned − reference {obj['paired']['aligned_vs_reference']['delta_em']:+.1%}, McNemar p = {obj['paired']['aligned_vs_reference']['mcnemar_exact_p']}).
- DPO: loss {dm['loss_first']:.3f} → {dm['loss_last']:.3f}; изменён {rep['dpo_effect']['answers_changed_vs_reference']} ответ(ов) из 64.
- Ручная оценка: {rep['n_complete']}/{rep['n_records']} записей, подробно — `blind_report.json`.
"""
(ARTIFACTS / 'REPRODUCE.md').write_text(REPRO, encoding='utf-8')

print(f"Полный режим: DPO {dm['budget']['done_optimizer_steps']} шагов, loss конечен, адаптер сохранён, reference не изменён")
print(f"Ответы: {len(blind_rows)} ID × 3 политики; форма: {sum(is_complete(r) for r in judged_rows)}/{len(judged_rows)} заполнено")
print(f"Обязательные артефакты: {len(REQUIRED)} есть; бонус: {BONUS_FILES or 'нет'}")
print(f"Версии: requirements-notebook.txt ({len(freeze.splitlines())} пакетов), version.json")
print(f"Перечень: submission_manifest.json ({len(files)} файлов); инструкции: REPRODUCE.md -> {ARTIFACTS}")

Полный режим: DPO 40 шагов, loss конечен, адаптер сохранён, reference не изменён
Ответы: 64 ID × 3 политики; форма: 64/64 заполнено
Обязательные артефакты: 15 есть; бонус: ['bonus.md', 'bonus_summary.json', 'bonus_logps.jsonl', 'bonus_predictions.jsonl']
Версии: requirements-notebook.txt (134 пакетов), version.json
Перечень: submission_manifest.json (29 файлов); инструкции: REPRODUCE.md -> artifacts\lab11


### Сдача и повторный запуск

- **Режим:** полный (`FAST_DEV_RUN=False`): 40 optimizer steps DPO × 8 пар, 64 тестовых ID × 3 политики. Ячейка выше проверяет это по `dpo_metrics.json`, `blind_predictions.jsonl` и заполненности `blind_judgements.jsonl`.
- **Конфигурация:**
  - `dpo_protocol.json` — β, lr, accumulation, шаги, порядок данных, формула loss;
  - `reference_policy.json` — источник reference, квантование, проверки;
  - `baseline_info.json` — декодирование;
  - `logprob_check.json` — разметка ответа и сдвиг next-token.
- **Версии зависимостей:** `requirements-notebook.txt` (полный `pip freeze` ядра `.venv`) и `version.json` (torch, transformers, peft, trl, bitsandbytes, GPU, ревизия датасета, sha256 адаптеров).
- **Артефакты** (`artifacts/lab11/`):
  - данные: `preferences.jsonl`, `pair_audit.json`;
  - обученный LoRA: `aligned_adapter/train/`;
  - обучение: `dpo_metrics.json`, `dpo_steps_train.jsonl`;
  - ответы: `baseline_predictions.jsonl`, `blind_predictions.jsonl`;
  - оценка: `objective_scores.json`, `blind_judgements.jsonl`, `blind_report.json`, `error_analysis.json`;
  - бонус: `bonus.md`, `bonus_summary.json`, `bonus_logps.jsonl`, `bonus_predictions.jsonl`.

  Перечень с размерами и SHA-256 — `submission_manifest.json`.
- **Инструкции повторного запуска:** `REPRODUCE.md` — окружение, данные, порядок ячеек, флаги `FORCE_RETRAIN` / `FORCE_REGENERATE` / `FORCE_REGENERATE_ALIGNED` / `REQUIRE_COMPLETE_FORM`, ключевые настройки, итоговые метрики.
- Все измерения получены реальными прогонами. Ручные суждения внесены оценщиком в форму; незаполненные значения обозначаются `pending` и в метрики не входят.